# Semantic Stroop: Final End-to-End Reproduction Notebook

This notebook reproduces the main results for:

**Priors Persist Through Suppression: A Stroop Paradigm for Lexical Override**

It is a unified public reproduction notebook. It is not an iteration log. It contains only the final analysis pipeline used for the manuscript:

1. **Behavioral external-validity sweep**
   - Qwen, Gemma, OLMo, and Mistral model settings.
   - 1B–2B anchor models and 7B/9B-class behavior-only models.
   - Antonym, arbitrary semantic, polysemy/entity, and domain-definition conflict families.
   - Game, glossary, technical-document, and scoped-definition prompt styles.
   - Full-sequence target-vs-distractor scoring.
   - Control regressions for lexical-prior strength, answer prior, tokenization, frequency proxy, prompt style, conflict type, model family, scale, and instruction tuning.

2. **Mechanistic analysis**
   - Aligned tractable modern model set: Qwen2.5-1.5B base/instruct, Gemma-2-2B base/instruct, OLMo-1B.
   - Source-triplet residual patching.
   - Source_all vs strongest source-position pair.
   - Split validation.
   - Random-pair controls.
   - Component-level writer/reader scans.
   - Reader-component zero-ablation mediation.
   - Target-vs-distractor logit decomposition.

The notebook saves CSVs, figures, Markdown summaries, and a final ZIP archive.

The definition-target swap control (Section 5.2 of the paper) is computed by this notebook as part of the mechanism pipeline. A self-contained supplementary notebook, `mechanism_random_controls_logit_decomp.ipynb`, regenerates the item-mismatched (random-source) control with per-perturbation per-item logit decomposition (used in Table 3 of the paper).


In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
import sys
import subprocess
import pkgutil

def ensure_package(pip_name, import_name=None):
    import_name = import_name or pip_name.replace("-", "_")
    if pkgutil.find_loader(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

required = [
    ("transformers>=4.45.0", "transformers"),
    ("accelerate", "accelerate"),
    ("sentencepiece", "sentencepiece"),
    ("protobuf", "google.protobuf"),
    ("transformer-lens>=3.0.0", "transformer_lens"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("statsmodels", "statsmodels"),
    ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"),
    ("tabulate", "tabulate"),
    ("wordfreq", "wordfreq"),
]

for pkg, imp in required:
    try:
        ensure_package(pkg, imp)
    except Exception as e:
        print("Dependency install failed or skipped:", pkg, repr(e))

print("Dependency setup complete.")

In [ ]:
# ============================================================
# 2. Imports, authentication, and global configuration
# ============================================================
import os
import gc
import json
import math
import random
import shutil
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy import stats
from IPython.display import display, Markdown

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformer_lens import HookedTransformer

try:
    import statsmodels.formula.api as smf
except Exception:
    smf = None

try:
    from wordfreq import zipf_frequency
except Exception:
    zipf_frequency = None

warnings.filterwarnings("ignore")
torch.set_grad_enabled(False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Behavior can safely use bfloat16 on A100-class GPUs.
BEHAVIOR_DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# Mechanistic analysis uses float32 to avoid bfloat16 -> NumPy/cache conversion issues.
MECHANISM_DTYPE = torch.float32

# ------------------------------
# Hugging Face authentication
# ------------------------------
# Option A: paste a private token here for a private Colab run.
# Do NOT commit/share a notebook containing a real token.
HF_TOKEN = ""  # e.g., "hf_xxx"

# Option B: Colab Secrets. Create a secret named HF_TOKEN.
try:
    from google.colab import userdata
    secret_token = userdata.get("HF_TOKEN")
    if secret_token and not HF_TOKEN:
        HF_TOKEN = secret_token
except Exception:
    pass

# Option C: environment variable.
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Logged into Hugging Face.")
    except Exception as e:
        print("Hugging Face login failed:", repr(e))
else:
    print("No Hugging Face token provided. Public models will run; gated models require authentication.")

# ------------------------------
# Runtime controls
# ------------------------------
RUN_BEHAVIOR = True
RUN_MECHANISM = True

FAST_DEBUG = False  # set True only for quick code-path tests

# Behavior settings.
MAX_ITEMS_PER_CONFLICT_TYPE = 80
NEUTRALS_PER_ITEM_BEHAVIOR = 6
BEHAVIOR_BATCH_SIZE = 8
BEHAVIOR_BOOTSTRAP_N = 3000
USE_CHAT_TEMPLATE_FOR_INSTRUCT = True

# Mechanism settings.
PATCH_N_ITEMS = 32
NEUTRALS_PER_ITEM_MECHANISM = 8
MECHANISM_BATCH_SIZE = 64
MECHANISM_BOOTSTRAP_N = 3000
N_SPLITS = 200
N_RANDOM_PERMUTATIONS = 50
# Optional targeted binding control: keep definition subject + query word matched,
# but swap only the definition-target activation from a donor item.
RUN_DEFINITION_TARGET_SWAP_CONTROL = True
N_DEFINITION_TARGET_SWAPS = 10

TOP_COMPONENTS = 20
MIN_MECHANISM_LAYER = 2
MIN_NONDEGENERATE_COMPONENT_LAYER = 2

SCAN_WRITERS = True
SCAN_READERS = True
RUN_MEDIATION_ABLATIONS = True

if FAST_DEBUG:
    MAX_ITEMS_PER_CONFLICT_TYPE = 8
    NEUTRALS_PER_ITEM_BEHAVIOR = 3
    BEHAVIOR_BATCH_SIZE = 4
    BEHAVIOR_BOOTSTRAP_N = 500
    PATCH_N_ITEMS = 8
    NEUTRALS_PER_ITEM_MECHANISM = 3
    MECHANISM_BOOTSTRAP_N = 500
    N_SPLITS = 20
    N_RANDOM_PERMUTATIONS = 5
    N_DEFINITION_TARGET_SWAPS = 3
    TOP_COMPONENTS = 5

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path(f"/content/semantic_stroop_final_reproduction_{RUN_ID}") if Path("/content").exists() else Path(f"./semantic_stroop_final_reproduction_{RUN_ID}")
CSV_DIR = OUT_DIR / "csv"
FIG_DIR = OUT_DIR / "figures"
SUMMARY_DIR = OUT_DIR / "summaries"

for d in [OUT_DIR, CSV_DIR, FIG_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)
print("Device:", DEVICE)
print("Behavior dtype:", BEHAVIOR_DTYPE)
print("Mechanism dtype:", MECHANISM_DTYPE)

In [ ]:
# ============================================================
# 3. Model specifications
# ============================================================

# Behavioral external-validity sweep.
# These are the model settings reported in the behavioral external-validity tables.
BEHAVIOR_MODEL_SPECS = [
    {"model_key": "qwen25_15b_base", "hf_id": "Qwen/Qwen2.5-1.5B", "family": "Qwen", "params_b": 1.5, "tuning": "base", "tier": "anchor_1b2b", "enabled": True},
    {"model_key": "qwen25_15b_instruct", "hf_id": "Qwen/Qwen2.5-1.5B-Instruct", "family": "Qwen", "params_b": 1.5, "tuning": "instruct", "tier": "anchor_1b2b", "enabled": True},
    {"model_key": "gemma2_2b_base", "hf_id": "google/gemma-2-2b", "family": "Gemma", "params_b": 2.0, "tuning": "base", "tier": "anchor_1b2b", "enabled": True},
    {"model_key": "gemma2_2b_instruct", "hf_id": "google/gemma-2-2b-it", "family": "Gemma", "params_b": 2.0, "tuning": "instruct", "tier": "anchor_1b2b", "enabled": True},
    {"model_key": "olmo_1b_base", "hf_id": "allenai/OLMo-1B-hf", "family": "OLMo", "params_b": 1.0, "tuning": "base", "tier": "anchor_1b2b", "enabled": True},

    {"model_key": "qwen25_7b_base", "hf_id": "Qwen/Qwen2.5-7B", "family": "Qwen", "params_b": 7.0, "tuning": "base", "tier": "large_7b9b", "enabled": True},
    {"model_key": "qwen25_7b_instruct", "hf_id": "Qwen/Qwen2.5-7B-Instruct", "family": "Qwen", "params_b": 7.0, "tuning": "instruct", "tier": "large_7b9b", "enabled": True},
    {"model_key": "gemma2_9b_base", "hf_id": "google/gemma-2-9b", "family": "Gemma", "params_b": 9.0, "tuning": "base", "tier": "large_7b9b", "enabled": True},
    {"model_key": "gemma2_9b_instruct", "hf_id": "google/gemma-2-9b-it", "family": "Gemma", "params_b": 9.0, "tuning": "instruct", "tier": "large_7b9b", "enabled": True},
    {"model_key": "mistral7b_base", "hf_id": "mistralai/Mistral-7B-v0.3", "family": "Mistral", "params_b": 7.0, "tuning": "base", "tier": "large_7b9b", "enabled": True},
    {"model_key": "mistral7b_instruct", "hf_id": "mistralai/Mistral-7B-Instruct-v0.3", "family": "Mistral", "params_b": 7.0, "tuning": "instruct", "tier": "large_7b9b", "enabled": True},
]

# Mechanistic analysis model set.
# These are the aligned tractable modern models used for activation patching and component scans.
MECHANISM_MODEL_SPECS = [
    {"model_key": "qwen25_15b_base", "hf_id": "Qwen/Qwen2.5-1.5B", "tl_name": "Qwen/Qwen2.5-1.5B", "family": "Qwen", "params_b": 1.5, "tuning": "base", "enabled": True},
    {"model_key": "qwen25_15b_instruct", "hf_id": "Qwen/Qwen2.5-1.5B-Instruct", "tl_name": "Qwen/Qwen2.5-1.5B-Instruct", "family": "Qwen", "params_b": 1.5, "tuning": "instruct", "enabled": True},
    {"model_key": "gemma2_2b_base", "hf_id": "google/gemma-2-2b", "tl_name": "google/gemma-2-2b", "family": "Gemma", "params_b": 2.0, "tuning": "base", "enabled": True},
    {"model_key": "gemma2_2b_instruct", "hf_id": "google/gemma-2-2b-it", "tl_name": "google/gemma-2-2b-it", "family": "Gemma", "params_b": 2.0, "tuning": "instruct", "enabled": True},
    {"model_key": "olmo_1b_base", "hf_id": "allenai/OLMo-1B-hf", "tl_name": "allenai/OLMo-1B-hf", "family": "OLMo", "params_b": 1.0, "tuning": "base", "enabled": True},
]

config = {
    "run_behavior": RUN_BEHAVIOR,
    "run_mechanism": RUN_MECHANISM,
    "fast_debug": FAST_DEBUG,
    "behavior_model_specs": BEHAVIOR_MODEL_SPECS,
    "mechanism_model_specs": MECHANISM_MODEL_SPECS,
    "max_items_per_conflict_type": MAX_ITEMS_PER_CONFLICT_TYPE,
    "neutrals_per_item_behavior": NEUTRALS_PER_ITEM_BEHAVIOR,
    "neutrals_per_item_mechanism": NEUTRALS_PER_ITEM_MECHANISM,
    "behavior_batch_size": BEHAVIOR_BATCH_SIZE,
    "mechanism_batch_size": MECHANISM_BATCH_SIZE,
    "behavior_bootstrap_n": BEHAVIOR_BOOTSTRAP_N,
    "mechanism_bootstrap_n": MECHANISM_BOOTSTRAP_N,
    "n_splits": N_SPLITS,
    "n_random_permutations": N_RANDOM_PERMUTATIONS,
    "top_components": TOP_COMPONENTS,
    "min_mechanism_layer": MIN_MECHANISM_LAYER,
    "min_nondegenerate_component_layer": MIN_NONDEGENERATE_COMPONENT_LAYER,
    "seed": SEED,
}

with open(OUT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Behavior models:", len(BEHAVIOR_MODEL_SPECS))
print("Mechanism models:", len(MECHANISM_MODEL_SPECS))

In [ ]:
# ============================================================
# 4. Shared utility functions
# ============================================================

def patch_tokenizer_no_bos_if_needed(tokenizer):
    """Disable BOS insertion when the tokenizer has no BOS token.

    This avoids failures in Qwen-family tokenizers where add_bos_token may be True
    while bos_token is None. Keeping explicit BOS off also makes prompt scoring
    consistent across model families.
    """
    try:
        if getattr(tokenizer, "bos_token", None) is None and hasattr(tokenizer, "add_bos_token"):
            tokenizer.add_bos_token = False
    except Exception:
        pass
    try:
        if getattr(tokenizer, "bos_token_id", None) is None and hasattr(tokenizer, "add_bos_token"):
            tokenizer.add_bos_token = False
    except Exception:
        pass
    try:
        if getattr(tokenizer, "pad_token", None) is None:
            tokenizer.pad_token = tokenizer.eos_token
    except Exception:
        pass
    return tokenizer

def load_safe_tokenizer(hf_id):
    attempts = [
        {"trust_remote_code": True, "use_fast": True, "add_bos_token": False},
        {"trust_remote_code": True, "use_fast": False, "add_bos_token": False},
        {"trust_remote_code": True, "use_fast": True},
        {"trust_remote_code": True, "use_fast": False},
    ]
    errors = []
    for kwargs in attempts:
        try:
            tok = AutoTokenizer.from_pretrained(hf_id, **kwargs)
            return patch_tokenizer_no_bos_if_needed(tok)
        except TypeError as e:
            errors.append(repr(e))
            kwargs = {k: v for k, v in kwargs.items() if k != "add_bos_token"}
            try:
                tok = AutoTokenizer.from_pretrained(hf_id, **kwargs)
                return patch_tokenizer_no_bos_if_needed(tok)
            except Exception as e2:
                errors.append(repr(e2))
        except Exception as e:
            errors.append(repr(e))
    raise RuntimeError(f"Failed to load tokenizer for {hf_id}: {errors[-3:]}")

def encode_prompt(tokenizer, prompt):
    tokenizer = patch_tokenizer_no_bos_if_needed(tokenizer)
    return tokenizer(prompt, add_special_tokens=False).input_ids

def continuation_ids(tokenizer, text):
    tokenizer = patch_tokenizer_no_bos_if_needed(tokenizer)
    return tokenizer(" " + str(text).strip(), add_special_tokens=False).input_ids

def maybe_chat_wrap(tokenizer, prompt, is_instruct):
    if USE_CHAT_TEMPLATE_FOR_INSTRUCT and is_instruct and hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            return prompt
    return prompt

def bootstrap_ci(vals, n_boot, seed=SEED):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        boots[i] = rng.choice(vals, size=len(vals), replace=True).mean()
    return float(vals.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

def summarize_values(vals, n_boot):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    mean, lo, hi = bootstrap_ci(vals, n_boot=n_boot)
    if len(vals) > 1:
        t, p = stats.ttest_1samp(vals, 0.0)
    else:
        t, p = np.nan, np.nan
    return pd.Series({
        "n": int(len(vals)),
        "mean": mean,
        "ci_low": lo,
        "ci_high": hi,
        "positive_fraction": float((vals > 0).mean()) if len(vals) else np.nan,
        "t": float(t) if np.isfinite(t) else np.nan,
        "p": float(p) if np.isfinite(p) else np.nan,
    })

def savefig(name):
    plt.tight_layout()
    path = FIG_DIR / name
    plt.savefig(path, dpi=250, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def safe_to_csv(df, name):
    path = CSV_DIR / name
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

## Behavioral external-validity experiment

This section constructs the final behavioral stimulus set and runs the full-sequence scoring sweep. It reproduces the manuscript's model-level, conflict-family, prompt-style, and controlled-regression behavioral results.

In [ ]:
# ============================================================
# 5. Behavioral stimuli
# ============================================================

ANTONYM_ITEMS = [
    ("small","tiny","big"),("large","big","small"),("hot","warm","cold"),("cold","chilly","hot"),
    ("happy","glad","sad"),("sad","unhappy","happy"),("fast","quick","slow"),("slow","sluggish","fast"),
    ("bright","shiny","dark"),("dark","dim","bright"),("hard","tough","soft"),("soft","gentle","hard"),
    ("strong","powerful","weak"),("weak","frail","strong"),("old","aged","new"),("new","fresh","old"),
    ("clean","neat","dirty"),("dirty","filthy","clean"),("rich","wealthy","poor"),("poor","needy","rich"),
    ("safe","secure","dangerous"),("dangerous","risky","safe"),("easy","simple","hard"),("difficult","hard","easy"),
    ("loud","noisy","quiet"),("quiet","silent","loud"),("clear","plain","confusing"),("confusing","unclear","clear"),
    ("empty","vacant","full"),("full","complete","empty"),("heavy","weighty","light"),("light","bright","dark"),
    ("narrow","thin","wide"),("wide","broad","narrow"),("deep","profound","shallow"),("shallow","superficial","deep"),
    ("early","prompt","late"),("late","tardy","early"),("near","close","far"),("far","distant","near"),
    ("high","tall","low"),("low","short","high"),("young","youthful","old"),("ancient","old","modern"),
    ("modern","new","ancient"),("rough","coarse","smooth"),("smooth","sleek","rough"),("dry","arid","wet"),
    ("wet","damp","dry"),("open","unlocked","closed"),("closed","shut","open"),("kind","nice","cruel"),
    ("cruel","mean","kind"),("brave","bold","cowardly"),("calm","peaceful","anxious"),("anxious","nervous","calm"),
    ("honest","truthful","dishonest"),("polite","courteous","rude"),("rude","impolite","polite"),
    ("loyal","faithful","disloyal"),("lazy","idle","active"),("active","busy","lazy"),("simple","plain","complex"),
    ("complex","complicated","simple"),("cheap","inexpensive","expensive"),("expensive","costly","cheap"),
    ("true","real","false"),("false","fake","true"),("thick","dense","thin"),("thin","slim","thick"),
    ("sharp","pointed","dull"),("dull","blunt","sharp"),("beautiful","pretty","ugly"),("ugly","hideous","beautiful"),
    ("smart","clever","stupid"),("stupid","dumb","smart"),("good","nice","bad"),("bad","poor","good"),
    ("correct","right","wrong"),("wrong","incorrect","right"),
]

ARBITRARY_ITEMS = [
    ("apple","fruit","river"),("banana","fruit","mountain"),("tiger","animal","pencil"),("doctor","hospital","forest"),
    ("teacher","school","ocean"),("piano","music","desert"),("bicycle","wheel","cloud"),("coffee","drink","island"),
    ("bread","food","engine"),("chair","furniture","planet"),("dog","animal","camera"),("cat","animal","window"),
    ("rose","flower","needle"),("car","vehicle","garden"),("train","vehicle","mirror"),("shoe","foot","castle"),
    ("bird","animal","hammer"),("book","reading","thunder"),("phone","call","valley"),("clock","time","butterfly"),
    ("knife","cutting","rainbow"),("shirt","clothing","volcano"),("moon","night","ladder"),("sun","day","blanket"),
    ("bank","money","sailor"),("king","royalty","battery"),("queen","royalty","harbor"),("mouse","animal","keyboard"),
    ("paper","writing","tunnel"),("camera","photo","village"),("fish","water","lantern"),("horse","animal","pillow"),
    ("milk","drink","bridge"),("city","urban","feather"),("forest","trees","clock"),("ocean","water","helmet"),
]

POLYSEMY_ITEMS = [
    ("apple","iPhone","music"),("python","snake","programming"),("java","coffee","programming"),("jaguar","animal","car"),
    ("mercury","planet","metal"),("bass","fish","music"),("crane","bird","machine"),("seal","animal","stamp"),
    ("spring","season","coil"),("delta","river","change"),("cell","biology","spreadsheet"),("table","furniture","data"),
    ("mouse","animal","computer"),("virus","disease","software"),("port","harbor","network"),("cookie","dessert","browser"),
    ("shell","beach","terminal"),("thread","sewing","computing"),("kernel","seed","operating"),("cloud","sky","computing"),
    ("token","coin","language"),("model","fashion","machine"),("branch","tree","git"),("commit","promise","git"),
    ("class","school","programming"),("object","thing","programming"),("method","procedure","programming"),("field","farm","database"),
    ("key","lock","keyboard"),("stream","river","data"),
]

DOMAIN_ITEMS = [
    ("port","harbor","socket"),("thread","sewing","process"),("cell","biology","spreadsheet"),
    ("token","coin","symbol"),("driver","person","software"),("kernel","seed","system"),
    ("class","school","type"),("object","thing","instance"),("field","farm","column"),
    ("branch","tree","version"),("commit","promise","snapshot"),("shell","beach","terminal"),
    ("pipe","plumbing","stream"),("queue","line","buffer"),("stack","pile","memory"),
    ("heap","pile","memory"),("cache","storage","memory"),("node","knot","vertex"),
    ("edge","border","link"),("root","plant","admin"),("leaf","tree","node"),
    ("parent","family","node"),("child","family","node"),("key","lock","identifier"),
    ("value","worth","data"),("index","book","position"),("pointer","finger","address"),
    ("array","display","list"),("string","thread","text"),("float","water","number"),
]

NEUTRAL_WORDS = [
    "thing","object","item","label","symbol","marker","term","name","token","entity","unit","case",
    "entry","sample","sign","tag","code","note","point","slot","type","class","record","field",
    "node","file","value","key","piece","part","form","line","phrase","signal","index","number",
    "letter","word","text","rule","box","shape","mark","group","set","map","link","path","step",
    "place","zone","area","page","cell","row","column","frame","scene","view","image"
]

def make_behavior_df(items, conflict_type, relation):
    df = pd.DataFrame(items, columns=["word", "distractor", "target"])
    df["conflict_type"] = conflict_type
    df["relation"] = relation
    return df

behavior_stimuli = pd.concat([
    make_behavior_df(ANTONYM_ITEMS, "antonym_remapping", "synonym"),
    make_behavior_df(ARBITRARY_ITEMS, "arbitrary_semantic_remapping", "related"),
    make_behavior_df(POLYSEMY_ITEMS, "polysemy_entity_remapping", "related"),
    make_behavior_df(DOMAIN_ITEMS, "domain_definition_remapping", "related"),
], ignore_index=True)

behavior_stimuli = (
    behavior_stimuli.groupby("conflict_type", group_keys=False)
    .head(MAX_ITEMS_PER_CONFLICT_TYPE)
    .reset_index(drop=True)
)
behavior_stimuli["item_id"] = np.arange(len(behavior_stimuli))

if zipf_frequency is not None:
    for col in ["word", "distractor", "target"]:
        behavior_stimuli[f"{col}_zipf"] = behavior_stimuli[col].apply(lambda x: float(zipf_frequency(str(x), "en")))
else:
    for col in ["word", "distractor", "target"]:
        behavior_stimuli[f"{col}_zipf"] = np.nan

behavior_stimuli["target_minus_distractor_zipf"] = behavior_stimuli["target_zipf"] - behavior_stimuli["distractor_zipf"]

PROMPT_WRAPPERS = [
    {
        "prompt_style": "game_rule",
        "template": "Complete the answer with one word or phrase only.\nIn this game, {subject} means {meaning}.\nQuestion: In this game, {relation_phrase} {query} is\nAnswer:",
    },
    {
        "prompt_style": "glossary",
        "template": "A glossary for this document defines \"{subject}\" as \"{meaning}\".\nUsing only this glossary, {relation_phrase} \"{query}\" is",
    },
    {
        "prompt_style": "technical_document",
        "template": "In the following technical document, the term \"{subject}\" refers to \"{meaning}\".\nAccording to the document, {relation_phrase} \"{query}\" is",
    },
    {
        "prompt_style": "definition_scope",
        "template": "For this passage only, interpret \"{subject}\" as \"{meaning}\".\nUnder this definition, {relation_phrase} \"{query}\" is",
    },
]

safe_to_csv(behavior_stimuli, "behavior_external_validity_stimuli.csv")
display(behavior_stimuli.groupby("conflict_type").size().reset_index(name="n_items"))
display(behavior_stimuli.head())

In [ ]:
# ============================================================
# 6. Behavioral experiment functions
# ============================================================

def relation_phrase(relation):
    return "a synonym of" if relation == "synonym" else "a word related to"

def make_behavior_prompt(template, subject, meaning, query, relation):
    return template.format(
        subject=subject,
        meaning=meaning,
        query=query,
        relation_phrase=relation_phrase(relation),
    )

def choose_behavior_neutrals(item_idx, item):
    blocked = {item["word"], item["distractor"], item["target"]}
    candidates = [w for w in NEUTRAL_WORDS if w not in blocked]
    offset = (item_idx * NEUTRALS_PER_ITEM_BEHAVIOR) % len(candidates)
    out = []
    for j in range(len(candidates)):
        candidate = candidates[(offset + j) % len(candidates)]
        if candidate not in out:
            out.append(candidate)
        if len(out) >= NEUTRALS_PER_ITEM_BEHAVIOR:
            break
    return out

def load_behavior_model_and_tokenizer(spec):
    tok = load_safe_tokenizer(spec["hf_id"])
    model = AutoModelForCausalLM.from_pretrained(
        spec["hf_id"],
        torch_dtype=BEHAVIOR_DTYPE,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    return model, tok

def score_continuations(model, tokenizer, prompts, continuations, batch_size=BEHAVIOR_BATCH_SIZE):
    rows = []
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

    for start in tqdm(range(0, len(prompts), batch_size), leave=False):
        end = min(start + batch_size, len(prompts))
        full_ids, prompt_lens, cont_lens = [], [], []

        for p, c in zip(prompts[start:end], continuations[start:end]):
            p_ids = encode_prompt(tokenizer, p)
            c_ids = continuation_ids(tokenizer, c)
            ids = p_ids + c_ids
            full_ids.append(ids)
            prompt_lens.append(len(p_ids))
            cont_lens.append(len(c_ids))

        max_len = max(len(x) for x in full_ids)
        input_ids = torch.full((len(full_ids), max_len), int(pad_id), dtype=torch.long, device=model.device)
        attn = torch.zeros((len(full_ids), max_len), dtype=torch.long, device=model.device)

        for i, ids in enumerate(full_ids):
            input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=model.device)
            attn[i, :len(ids)] = 1

        with torch.inference_mode():
            out = model(input_ids=input_ids, attention_mask=attn)
            logits = out.logits.float()
            logprobs = torch.log_softmax(logits, dim=-1)

        for i in range(len(full_ids)):
            p_len, c_len = prompt_lens[i], cont_lens[i]
            vals = []
            for j in range(c_len):
                pos = p_len + j
                tok = int(input_ids[i, pos].item())
                vals.append(float(logprobs[i, pos - 1, tok].detach().cpu()))
            rows.append({
                "sum_logprob": float(np.sum(vals)),
                "mean_logprob": float(np.mean(vals)) if vals else np.nan,
                "n_cont_tokens": int(c_len),
            })

        del input_ids, attn, out, logits, logprobs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)

def build_behavior_requests_for_model(spec, tokenizer):
    rows = []
    is_instruct = spec["tuning"] == "instruct"

    for _, item in behavior_stimuli.iterrows():
        item_id = int(item["item_id"])
        neutrals = choose_behavior_neutrals(item_id, item)

        for wrapper in PROMPT_WRAPPERS:
            prompt_style = wrapper["prompt_style"]
            template = wrapper["template"]

            raw_prompt = make_behavior_prompt(template, item["word"], item["target"], item["word"], item["relation"])
            prompt = maybe_chat_wrap(tokenizer, raw_prompt, is_instruct)

            for role, cont in [("target", item["target"]), ("distractor", item["distractor"])]:
                rows.append({
                    **spec,
                    "item_id": item_id,
                    "word": item["word"],
                    "target": item["target"],
                    "distractor": item["distractor"],
                    "conflict_type": item["conflict_type"],
                    "relation": item["relation"],
                    "prompt_style": prompt_style,
                    "condition": "conflict",
                    "neutral_word": "",
                    "probe": "main",
                    "role": role,
                    "continuation": cont,
                    "prompt": prompt,
                })

            for neutral in neutrals:
                raw_prompt = make_behavior_prompt(template, neutral, item["target"], neutral, item["relation"])
                prompt = maybe_chat_wrap(tokenizer, raw_prompt, is_instruct)
                for role, cont in [("target", item["target"]), ("distractor", item["distractor"])]:
                    rows.append({
                        **spec,
                        "item_id": item_id,
                        "word": item["word"],
                        "target": item["target"],
                        "distractor": item["distractor"],
                        "conflict_type": item["conflict_type"],
                        "relation": item["relation"],
                        "prompt_style": prompt_style,
                        "condition": "neutral",
                        "neutral_word": neutral,
                        "probe": "main",
                        "role": role,
                        "continuation": cont,
                        "prompt": prompt,
                    })

        # Lexical-prior probe.
        raw = f"A synonym of {item['word']} is" if item["relation"] == "synonym" else f"A word related to {item['word']} is"
        prompt = maybe_chat_wrap(tokenizer, raw, is_instruct)
        for role, cont in [("target", item["target"]), ("distractor", item["distractor"])]:
            rows.append({
                **spec,
                "item_id": item_id,
                "word": item["word"],
                "target": item["target"],
                "distractor": item["distractor"],
                "conflict_type": item["conflict_type"],
                "relation": item["relation"],
                "prompt_style": "lexical_prior",
                "condition": "probe",
                "neutral_word": "",
                "probe": "lexical_prior",
                "role": role,
                "continuation": cont,
                "prompt": prompt,
            })

        # Answer-prior probe.
        prompt = maybe_chat_wrap(tokenizer, "Answer:", is_instruct)
        for role, cont in [("target", item["target"]), ("distractor", item["distractor"])]:
            rows.append({
                **spec,
                "item_id": item_id,
                "word": item["word"],
                "target": item["target"],
                "distractor": item["distractor"],
                "conflict_type": item["conflict_type"],
                "relation": item["relation"],
                "prompt_style": "answer_prior",
                "condition": "probe",
                "neutral_word": "",
                "probe": "answer_prior",
                "role": role,
                "continuation": cont,
                "prompt": prompt,
            })

    return pd.DataFrame(rows)

def analyze_behavior_model(spec):
    print("\n" + "=" * 100)
    print("Behavior model:", spec["model_key"], spec["hf_id"])
    print("=" * 100)

    model, tokenizer = load_behavior_model_and_tokenizer(spec)
    req = build_behavior_requests_for_model(spec, tokenizer)
    print("Requests:", len(req))

    sc = score_continuations(
        model,
        tokenizer,
        req["prompt"].tolist(),
        req["continuation"].tolist(),
        batch_size=BEHAVIOR_BATCH_SIZE,
    )
    scored = pd.concat([req.reset_index(drop=True), sc.reset_index(drop=True)], axis=1)

    pivot_cols = [
        "model_key", "hf_id", "family", "params_b", "tuning", "tier",
        "item_id", "word", "target", "distractor", "conflict_type", "relation",
        "prompt_style", "condition", "neutral_word", "probe",
    ]

    wide = scored.pivot_table(
        index=pivot_cols,
        columns="role",
        values=["sum_logprob", "mean_logprob", "n_cont_tokens"],
        aggfunc="first",
    ).reset_index()
    wide.columns = ["_".join([str(x) for x in col if str(x) != ""]) for col in wide.columns]
    wide["score_sum_logprob"] = wide["sum_logprob_target"] - wide["sum_logprob_distractor"]
    wide["score_mean_logprob"] = wide["mean_logprob_target"] - wide["mean_logprob_distractor"]

    # Main effect rows.
    rows = []
    main = wide[wide["probe"] == "main"].copy()
    for (item_id, prompt_style), sub in main.groupby(["item_id", "prompt_style"], sort=False):
        conflict = sub[sub["condition"] == "conflict"]
        neutral = sub[sub["condition"] == "neutral"]
        if len(conflict) != 1 or len(neutral) == 0:
            continue
        conflict = conflict.iloc[0]
        for mode in ["sum_logprob", "mean_logprob"]:
            col = f"score_{mode}"
            rows.append({
                "model_key": spec["model_key"],
                "hf_id": spec["hf_id"],
                "family": spec["family"],
                "params_b": spec["params_b"],
                "tuning": spec["tuning"],
                "tier": spec["tier"],
                "item_id": int(item_id),
                "word": conflict["word"],
                "target": conflict["target"],
                "distractor": conflict["distractor"],
                "conflict_type": conflict["conflict_type"],
                "relation": conflict["relation"],
                "prompt_style": prompt_style,
                "score_mode": mode,
                "conflict_score": float(conflict[col]),
                "neutral_mean": float(neutral[col].mean()),
                "neutral_std": float(neutral[col].std(ddof=1)),
                "stroop_effect": float(neutral[col].mean() - conflict[col]),
            })
    effects = pd.DataFrame(rows)

    # Probe scores.
    probes = wide[wide["probe"].isin(["lexical_prior", "answer_prior"])].copy()
    probe_piv = probes.pivot_table(
        index=["item_id", "probe"],
        values=["score_sum_logprob", "score_mean_logprob"],
        aggfunc="first",
    ).reset_index()
    probe_wide = probe_piv.pivot(
        index="item_id",
        columns="probe",
        values=["score_sum_logprob", "score_mean_logprob"],
    ).reset_index()
    probe_wide.columns = ["_".join([str(x) for x in col if str(x) != ""]) for col in probe_wide.columns]
    probe_wide = probe_wide.rename(columns={
        "item_id_": "item_id",
        "score_sum_logprob_lexical_prior": "lexical_target_minus_distractor_sum",
        "score_mean_logprob_lexical_prior": "lexical_target_minus_distractor_mean",
        "score_sum_logprob_answer_prior": "answer_prior_target_minus_distractor_sum",
        "score_mean_logprob_answer_prior": "answer_prior_target_minus_distractor_mean",
    })
    effects = effects.merge(probe_wide, on="item_id", how="left")
    effects["lexical_distractor_advantage_sum"] = -effects["lexical_target_minus_distractor_sum"]
    effects["lexical_distractor_advantage_mean"] = -effects["lexical_target_minus_distractor_mean"]

    # Tokenization metadata.
    tok_rows = []
    for _, item in behavior_stimuli.iterrows():
        tok_rows.append({
            "item_id": int(item["item_id"]),
            "word_n_tokens": len(continuation_ids(tokenizer, item["word"])),
            "target_n_tokens": len(continuation_ids(tokenizer, item["target"])),
            "distractor_n_tokens": len(continuation_ids(tokenizer, item["distractor"])),
        })
    tok_df = pd.DataFrame(tok_rows)
    tok_df["target_minus_distractor_tokens"] = tok_df["target_n_tokens"] - tok_df["distractor_n_tokens"]
    tok_df["is_single_token_pair"] = (tok_df["target_n_tokens"] == 1) & (tok_df["distractor_n_tokens"] == 1)

    meta_cols = [
        "item_id", "word_zipf", "target_zipf", "distractor_zipf",
        "target_minus_distractor_zipf",
    ]
    effects = effects.merge(behavior_stimuli[meta_cols], on="item_id", how="left")
    effects = effects.merge(tok_df, on="item_id", how="left")

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return wide, effects

In [ ]:
# ============================================================
# 7. Run behavioral external-validity sweep
# ============================================================
behavior_run_log = []
behavior_effects_all = []
behavior_scores_all = []

if RUN_BEHAVIOR:
    for spec in BEHAVIOR_MODEL_SPECS:
        if not spec.get("enabled", True):
            continue
        try:
            wide, effects = analyze_behavior_model(spec)
            behavior_scores_all.append(wide)
            behavior_effects_all.append(effects)
            wide.to_csv(CSV_DIR / f"behavior_condition_scores_{spec['model_key']}.csv", index=False)
            effects.to_csv(CSV_DIR / f"behavior_effects_{spec['model_key']}.csv", index=False)
            behavior_run_log.append({**spec, "status": "success", "error": "", "n_effect_rows": len(effects)})
        except Exception as e:
            print("Behavior model failed:", spec["model_key"], repr(e))
            behavior_run_log.append({**spec, "status": "error", "error": repr(e), "n_effect_rows": np.nan})
            if not FAST_DEBUG:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
else:
    print("RUN_BEHAVIOR is False; skipping behavioral sweep.")

behavior_run_log_df = pd.DataFrame(behavior_run_log)
behavior_effects_df = pd.concat(behavior_effects_all, ignore_index=True) if behavior_effects_all else pd.DataFrame()
behavior_scores_df = pd.concat(behavior_scores_all, ignore_index=True) if behavior_scores_all else pd.DataFrame()

safe_to_csv(behavior_run_log_df, "behavior_model_run_log.csv")
if len(behavior_effects_df):
    safe_to_csv(behavior_effects_df, "behavior_item_prompt_effects.csv")
if len(behavior_scores_df):
    safe_to_csv(behavior_scores_df, "behavior_condition_and_probe_scores_compact.csv")

display(behavior_run_log_df)
display(behavior_effects_df.head() if len(behavior_effects_df) else behavior_effects_df)

In [ ]:
# ============================================================
# 8. Behavioral summaries, regressions, and figures
# ============================================================
if RUN_BEHAVIOR and len(behavior_effects_df):
    behavior_summary_model = behavior_effects_df.groupby(
        ["model_key", "family", "params_b", "tuning", "tier", "score_mode"],
        sort=False,
    ).apply(lambda d: summarize_values(d["stroop_effect"], BEHAVIOR_BOOTSTRAP_N)).reset_index()

    behavior_summary_type = behavior_effects_df.groupby(
        ["model_key", "family", "params_b", "tuning", "tier", "conflict_type", "score_mode"],
        sort=False,
    ).apply(lambda d: summarize_values(d["stroop_effect"], BEHAVIOR_BOOTSTRAP_N)).reset_index()

    behavior_summary_style = behavior_effects_df.groupby(
        ["model_key", "family", "params_b", "tuning", "tier", "prompt_style", "score_mode"],
        sort=False,
    ).apply(lambda d: summarize_values(d["stroop_effect"], BEHAVIOR_BOOTSTRAP_N)).reset_index()

    behavior_summary_type_style = behavior_effects_df.groupby(
        ["model_key", "family", "params_b", "tuning", "tier", "conflict_type", "prompt_style", "score_mode"],
        sort=False,
    ).apply(lambda d: summarize_values(d["stroop_effect"], BEHAVIOR_BOOTSTRAP_N)).reset_index()

    behavior_aggregate_type = behavior_effects_df.groupby(
        ["conflict_type", "score_mode"],
        sort=False,
    ).apply(lambda d: summarize_values(d["stroop_effect"], BEHAVIOR_BOOTSTRAP_N)).reset_index()

    behavior_aggregate_style = behavior_effects_df.groupby(
        ["prompt_style", "score_mode"],
        sort=False,
    ).apply(lambda d: summarize_values(d["stroop_effect"], BEHAVIOR_BOOTSTRAP_N)).reset_index()

    pair_rows = []
    for (family, params_b, score_mode), sub in behavior_summary_model.groupby(["family", "params_b", "score_mode"], sort=False):
        base = sub[sub["tuning"] == "base"]
        inst = sub[sub["tuning"] == "instruct"]
        if len(base) and len(inst):
            pair_rows.append({
                "family": family,
                "params_b": params_b,
                "score_mode": score_mode,
                "base_model": base["model_key"].iloc[0],
                "instruct_model": inst["model_key"].iloc[0],
                "base_mean": float(base["mean"].iloc[0]),
                "instruct_mean": float(inst["mean"].iloc[0]),
                "instruct_minus_base": float(inst["mean"].iloc[0] - base["mean"].iloc[0]),
            })
    behavior_base_vs_instruct = pd.DataFrame(pair_rows)

    # Controlled regressions.
    behavior_regression_rows = []
    if smf is not None:
        for mode in ["sum_logprob", "mean_logprob"]:
            data = behavior_effects_df[behavior_effects_df["score_mode"] == mode].copy()
            data["is_instruct"] = (data["tuning"] == "instruct").astype(int)
            data["log_params_b"] = np.log(data["params_b"].astype(float))
            lex_col = "lexical_distractor_advantage_sum" if mode == "sum_logprob" else "lexical_distractor_advantage_mean"
            ans_col = "answer_prior_target_minus_distractor_sum" if mode == "sum_logprob" else "answer_prior_target_minus_distractor_mean"
            data = data.rename(columns={lex_col: "lexical_advantage", ans_col: "answer_prior"})

            formulas = {
                "main_controls": "stroop_effect ~ lexical_advantage + answer_prior + target_minus_distractor_tokens + target_minus_distractor_zipf + is_instruct + log_params_b + C(conflict_type) + C(prompt_style) + C(family)",
                "type_interactions": "stroop_effect ~ lexical_advantage * C(conflict_type) + answer_prior + target_minus_distractor_tokens + target_minus_distractor_zipf + is_instruct + log_params_b + C(prompt_style) + C(family)",
                "prompt_interactions": "stroop_effect ~ lexical_advantage + answer_prior + target_minus_distractor_tokens + target_minus_distractor_zipf + is_instruct + log_params_b + C(conflict_type) * C(prompt_style) + C(family)",
            }

            for regression_name, formula in formulas.items():
                try:
                    fit = smf.ols(formula, data=data).fit(
                        cov_type="cluster",
                        cov_kwds={"groups": data["item_id"].astype(str)},
                    )
                    conf = fit.conf_int()
                    for term, coef in fit.params.items():
                        behavior_regression_rows.append({
                            "score_mode": mode,
                            "regression": regression_name,
                            "term": term,
                            "coef": float(coef),
                            "se": float(fit.bse[term]),
                            "p": float(fit.pvalues[term]),
                            "ci_low": float(conf.loc[term, 0]),
                            "ci_high": float(conf.loc[term, 1]),
                            "n": int(fit.nobs),
                        })
                except Exception as e:
                    behavior_regression_rows.append({
                        "score_mode": mode,
                        "regression": regression_name,
                        "term": "REGRESSION_FAILED",
                        "error": repr(e),
                    })
    behavior_regression_df = pd.DataFrame(behavior_regression_rows)

    # Robustness regressions for manuscript audit.
    # This table supports the statement that key signs are stable under:
    # item-clustered SEs, model-clustered SEs, two-way item/model clustered SEs,
    # and a mixed-effects specification with model random intercepts plus item variance components.
    behavior_regression_robustness_rows = []
    if smf is not None:
        key_terms = {"Intercept", "lexical_advantage", "is_instruct", "log_params_b"}
        for mode in ["sum_logprob", "mean_logprob"]:
            data = behavior_effects_df[behavior_effects_df["score_mode"] == mode].copy()
            data["is_instruct"] = (data["tuning"] == "instruct").astype(int)
            data["log_params_b"] = np.log(data["params_b"].astype(float))
            lex_col = "lexical_distractor_advantage_sum" if mode == "sum_logprob" else "lexical_distractor_advantage_mean"
            ans_col = "answer_prior_target_minus_distractor_sum" if mode == "sum_logprob" else "answer_prior_target_minus_distractor_mean"
            data = data.rename(columns={lex_col: "lexical_advantage", ans_col: "answer_prior"})

            formula = "stroop_effect ~ lexical_advantage + answer_prior + target_minus_distractor_tokens + target_minus_distractor_zipf + is_instruct + log_params_b + C(conflict_type) + C(prompt_style) + C(family)"

            item_codes = pd.factorize(data["item_id"].astype(str))[0]
            model_codes = pd.factorize(data["model_key"].astype(str))[0]
            two_way_groups = np.column_stack([item_codes, model_codes])

            ols_specs = [
                ("cluster_by_item", {"cov_type": "cluster", "cov_kwds": {"groups": data["item_id"].astype(str)}}),
                ("cluster_by_model", {"cov_type": "cluster", "cov_kwds": {"groups": data["model_key"].astype(str)}}),
                ("two_way_cluster_item_model", {"cov_type": "cluster", "cov_kwds": {"groups": two_way_groups}}),
            ]

            for spec_name, fit_kwargs in ols_specs:
                try:
                    fit = smf.ols(formula, data=data).fit(**fit_kwargs)
                    conf = fit.conf_int()
                    for term, coef in fit.params.items():
                        if term in key_terms:
                            behavior_regression_robustness_rows.append({
                                "score_mode": mode,
                                "specification": spec_name,
                                "term": term,
                                "coef": float(coef),
                                "se": float(fit.bse[term]),
                                "p": float(fit.pvalues[term]),
                                "ci_low": float(conf.loc[term, 0]),
                                "ci_high": float(conf.loc[term, 1]),
                                "n": int(fit.nobs),
                                "status": "success",
                                "error": "",
                            })
                except Exception as e:
                    behavior_regression_robustness_rows.append({
                        "score_mode": mode,
                        "specification": spec_name,
                        "term": "SPECIFICATION_FAILED",
                        "coef": np.nan,
                        "se": np.nan,
                        "p": np.nan,
                        "ci_low": np.nan,
                        "ci_high": np.nan,
                        "n": len(data),
                        "status": "error",
                        "error": repr(e),
                    })

            try:
                md = smf.mixedlm(
                    formula,
                    data=data,
                    groups=data["model_key"].astype(str),
                    vc_formula={"item": "0 + C(item_id)"},
                )
                mfit = md.fit(reml=False, method="lbfgs", maxiter=500, disp=False)
                conf = mfit.conf_int()
                for term, coef in mfit.params.items():
                    if term in key_terms:
                        behavior_regression_robustness_rows.append({
                            "score_mode": mode,
                            "specification": "mixed_effects_item_model_random_intercepts",
                            "term": term,
                            "coef": float(coef),
                            "se": float(mfit.bse[term]) if term in mfit.bse.index else np.nan,
                            "p": float(mfit.pvalues[term]) if term in mfit.pvalues.index else np.nan,
                            "ci_low": float(conf.loc[term, 0]) if term in conf.index else np.nan,
                            "ci_high": float(conf.loc[term, 1]) if term in conf.index else np.nan,
                            "n": int(mfit.nobs),
                            "status": "success",
                            "error": "",
                        })
            except Exception as e:
                behavior_regression_robustness_rows.append({
                    "score_mode": mode,
                    "specification": "mixed_effects_item_model_random_intercepts",
                    "term": "SPECIFICATION_FAILED",
                    "coef": np.nan,
                    "se": np.nan,
                    "p": np.nan,
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                    "n": len(data),
                    "status": "error",
                    "error": repr(e),
                })

    behavior_regression_robustness_df = pd.DataFrame(behavior_regression_robustness_rows)


    # Evidence checklist.
    check_rows = []
    for mode in ["sum_logprob", "mean_logprob"]:
        sm = behavior_summary_model[behavior_summary_model["score_mode"] == mode]
        st = behavior_summary_type[behavior_summary_type["score_mode"] == mode]
        ss = behavior_summary_style[behavior_summary_style["score_mode"] == mode]
        reg = behavior_regression_df[
            (behavior_regression_df["score_mode"] == mode) &
            (behavior_regression_df["regression"] == "main_controls") &
            (behavior_regression_df["term"] == "lexical_advantage")
        ]
        intercept = behavior_regression_df[
            (behavior_regression_df["score_mode"] == mode) &
            (behavior_regression_df["regression"] == "main_controls") &
            (behavior_regression_df["term"] == "Intercept")
        ]

        check_rows.append({
            "area": "model_level_effects",
            "score_mode": mode,
            "passed": bool(len(sm) and (sm["ci_low"] > 0).all()),
            "evidence": f"{int((sm['ci_low'] > 0).sum())}/{len(sm)} successful model-level CIs > 0",
        })
        check_rows.append({
            "area": "conflict_type_external_validity",
            "score_mode": mode,
            "passed": bool(len(st) and (st["ci_low"] > 0).mean() >= 0.80),
            "evidence": f"{int((st['ci_low'] > 0).sum())}/{len(st)} model-conflict-type cells have CI > 0",
        })
        check_rows.append({
            "area": "document_style_prompt_validity",
            "score_mode": mode,
            "passed": bool(len(ss) and (ss["ci_low"] > 0).mean() >= 0.80),
            "evidence": f"{int((ss['ci_low'] > 0).sum())}/{len(ss)} model-prompt-style cells have CI > 0",
        })
        check_rows.append({
            "area": "lexical_prior_strength_predicts_interference",
            "score_mode": mode,
            "passed": bool(len(reg) and reg["ci_low"].iloc[0] > 0),
            "evidence": reg.to_dict("records")[:1] if len(reg) else "missing",
        })
        check_rows.append({
            "area": "controlled_intercept_positive",
            "score_mode": mode,
            "passed": bool(len(intercept) and intercept["ci_low"].iloc[0] > 0),
            "evidence": intercept.to_dict("records")[:1] if len(intercept) else "missing",
        })
    behavior_checklist_df = pd.DataFrame(check_rows)

    # Save tables.
    safe_to_csv(behavior_summary_model, "behavior_summary_by_model.csv")
    safe_to_csv(behavior_summary_type, "behavior_summary_by_model_conflict_type.csv")
    safe_to_csv(behavior_summary_style, "behavior_summary_by_model_prompt_style.csv")
    safe_to_csv(behavior_summary_type_style, "behavior_summary_by_model_type_style.csv")
    safe_to_csv(behavior_aggregate_type, "behavior_aggregate_by_conflict_type.csv")
    safe_to_csv(behavior_aggregate_style, "behavior_aggregate_by_prompt_style.csv")
    safe_to_csv(behavior_base_vs_instruct, "behavior_base_vs_instruct.csv")
    safe_to_csv(behavior_regression_df, "behavior_control_regressions.csv")
    safe_to_csv(behavior_regression_robustness_df, "behavior_regression_robustness.csv")
    safe_to_csv(behavior_checklist_df, "behavior_evidence_checklist.csv")

    display(behavior_summary_model)
    display(behavior_aggregate_type)
    display(behavior_aggregate_style)
    display(behavior_checklist_df)
    display(behavior_regression_robustness_df.head(20))

    # Figures.
    for mode in ["sum_logprob", "mean_logprob"]:
        plot = behavior_summary_model[behavior_summary_model["score_mode"] == mode].sort_values(["tier", "family", "params_b", "tuning"])
        if len(plot):
            y = np.arange(len(plot))
            plt.figure(figsize=(10, max(4, 0.38 * len(plot) + 2)))
            plt.errorbar(
                plot["mean"], y,
                xerr=[plot["mean"] - plot["ci_low"], plot["ci_high"] - plot["mean"]],
                fmt="o",
                capsize=3,
            )
            plt.axvline(0, linewidth=1)
            labels = plot.apply(lambda r: f"{r['model_key']} ({r['tier']})", axis=1)
            plt.yticks(y, labels, fontsize=8)
            plt.xlabel(f"Semantic Stroop effect ({mode})")
            plt.title(f"Model-level behavioral sweep ({mode})")
            savefig(f"behavior_model_forest_{mode}.png")

        pivot = behavior_summary_type[behavior_summary_type["score_mode"] == mode].pivot_table(
            index="model_key", columns="conflict_type", values="mean", aggfunc="mean"
        )
        if len(pivot):
            plt.figure(figsize=(10, max(4, 0.35 * len(pivot) + 2)))
            plt.imshow(pivot.values, aspect="auto")
            plt.colorbar(label=f"Effect ({mode})")
            plt.xticks(range(len(pivot.columns)), [c.replace("_", "\n") for c in pivot.columns], fontsize=8)
            plt.yticks(range(len(pivot.index)), pivot.index, fontsize=8)
            plt.title(f"Effect by conflict family ({mode})")
            savefig(f"behavior_conflict_type_heatmap_{mode}.png")

        pivot = behavior_summary_style[behavior_summary_style["score_mode"] == mode].pivot_table(
            index="model_key", columns="prompt_style", values="mean", aggfunc="mean"
        )
        if len(pivot):
            plt.figure(figsize=(10, max(4, 0.35 * len(pivot) + 2)))
            plt.imshow(pivot.values, aspect="auto")
            plt.colorbar(label=f"Effect ({mode})")
            plt.xticks(range(len(pivot.columns)), [c.replace("_", "\n") for c in pivot.columns], fontsize=8)
            plt.yticks(range(len(pivot.index)), pivot.index, fontsize=8)
            plt.title(f"Effect by prompt style ({mode})")
            savefig(f"behavior_prompt_style_heatmap_{mode}.png")

        data = behavior_effects_df[behavior_effects_df["score_mode"] == mode].copy()
        if len(data):
            lex_col = "lexical_distractor_advantage_sum" if mode == "sum_logprob" else "lexical_distractor_advantage_mean"
            plt.figure(figsize=(6, 5))
            plt.scatter(data[lex_col], data["stroop_effect"], alpha=0.12)
            plt.axhline(0, linewidth=1)
            plt.axvline(0, linewidth=1)
            plt.xlabel("Lexical-prior distractor advantage")
            plt.ylabel(f"Semantic Stroop effect ({mode})")
            plt.title("Lexical-prior strength predicts interference")
            savefig(f"behavior_lexical_prior_vs_effect_{mode}.png")

        plot = behavior_base_vs_instruct[behavior_base_vs_instruct["score_mode"] == mode] if len(behavior_base_vs_instruct) else pd.DataFrame()
        if len(plot):
            x = np.arange(len(plot))
            plt.figure(figsize=(8, 4))
            plt.bar(x - 0.18, plot["base_mean"], width=0.36, label="base")
            plt.bar(x + 0.18, plot["instruct_mean"], width=0.36, label="instruct")
            plt.axhline(0, linewidth=1)
            plt.xticks(x, plot["family"] + " " + plot["params_b"].astype(str) + "B", rotation=30, ha="right")
            plt.ylabel(f"Effect ({mode})")
            plt.title("Base vs instruction-tuned models")
            plt.legend()
            savefig(f"behavior_base_vs_instruct_{mode}.png")
else:
    print("Behavioral summaries skipped.")

## Mechanistic experiment

This section reproduces the causal mechanism results on the aligned tractable model set. It uses TransformerLens for activation patching and component-level scans. Mechanism runs use float32 by design to avoid cache/logit conversion issues.

In [ ]:
# ============================================================
# 9. Mechanistic stimuli and helpers
# ============================================================

MECHANISM_ITEMS = [
    ("small","tiny","big"),("large","big","small"),("hot","warm","cold"),("cold","chilly","hot"),
    ("happy","glad","sad"),("sad","unhappy","happy"),("fast","quick","slow"),("slow","sluggish","fast"),
    ("bright","shiny","dark"),("dark","dim","bright"),("hard","tough","soft"),("soft","gentle","hard"),
    ("strong","powerful","weak"),("weak","frail","strong"),("old","aged","new"),("new","fresh","old"),
    ("clean","neat","dirty"),("dirty","filthy","clean"),("rich","wealthy","poor"),("poor","needy","rich"),
    ("safe","secure","dangerous"),("dangerous","risky","safe"),("easy","simple","hard"),("difficult","hard","easy"),
    ("loud","noisy","quiet"),("quiet","silent","loud"),("clear","plain","confusing"),("confusing","unclear","clear"),
    ("empty","vacant","full"),("full","complete","empty"),("heavy","weighty","light"),("light","bright","dark"),
    ("narrow","thin","wide"),("wide","broad","narrow"),("deep","profound","shallow"),("shallow","superficial","deep"),
    ("near","close","far"),("far","distant","near"),("young","youthful","old"),("rough","coarse","smooth"),
    ("smooth","sleek","rough"),("wet","damp","dry"),("open","unlocked","closed"),("closed","shut","open"),
    ("kind","nice","cruel"),("cruel","mean","kind"),("brave","bold","cowardly"),("calm","peaceful","anxious"),
    ("anxious","nervous","calm"),("honest","truthful","dishonest"),("lazy","idle","active"),("active","busy","lazy"),
    ("complex","complicated","simple"),("cheap","inexpensive","expensive"),("expensive","costly","cheap"),
    ("false","fake","true"),("thick","dense","thin"),("thin","slim","thick"),("sharp","pointed","dull"),
    ("dull","blunt","sharp"),("ugly","hideous","beautiful"),("smart","clever","stupid"),("stupid","dumb","smart"),
    ("bad","poor","good"),("wrong","incorrect","right"),
]

mechanism_stimuli = pd.DataFrame(MECHANISM_ITEMS, columns=["word", "distractor", "target"])
mechanism_stimuli["item_id"] = np.arange(len(mechanism_stimuli))
safe_to_csv(mechanism_stimuli, "mechanism_stimuli.csv")

MECHANISM_TEMPLATE = "Complete the answer with one word only.\nIn this game, {subject} means {meaning}.\nQuestion: In this game, a synonym of {query} is\nAnswer:"

def mechanism_prompt(subject, meaning, query):
    return MECHANISM_TEMPLATE.format(subject=subject, meaning=meaning, query=query)

def choose_mechanism_neutrals(item_idx, item):
    blocked = {item["word"], item["distractor"], item["target"]}
    candidates = [w for w in NEUTRAL_WORDS if w not in blocked]
    offset = (item_idx * NEUTRALS_PER_ITEM_MECHANISM) % len(candidates)
    out = []
    for j in range(len(candidates)):
        candidate = candidates[(offset + j) % len(candidates)]
        if candidate not in out:
            out.append(candidate)
        if len(out) >= NEUTRALS_PER_ITEM_MECHANISM:
            break
    return out

SOURCE_SINGLETON_GROUPS = ["definition_subject", "definition_target", "query_word"]
SOURCE_PAIR_GROUPS = ["definition_subject__definition_target", "definition_subject__query_word", "definition_target__query_word"]
SOURCE_COMBO_GROUPS = SOURCE_SINGLETON_GROUPS + SOURCE_PAIR_GROUPS + ["source_all"]
RESIDUAL_COMPONENTS = ["hook_resid_pre", "hook_resid_mid", "hook_resid_post"]

def find_subsequence(sequence, subseq):
    starts = []
    for i in range(len(sequence) - len(subseq) + 1):
        if list(sequence[i:i+len(subseq)]) == list(subseq):
            starts.append(i)
    return starts

def expand_positions(starts, length):
    out = []
    for s in starts:
        out.extend(range(s, s + length))
    return sorted(set(out))

def patch_label(layer, component, group):
    return f"{component}_L{layer}_{group}"

def recovery(clean, corrupt, patched, idx=None):
    clean = np.asarray(clean, dtype=float)
    corrupt = np.asarray(corrupt, dtype=float)
    patched = np.asarray(patched, dtype=float)
    if idx is None:
        idx = np.arange(len(clean))
    idx = np.asarray(idx)
    denom = clean[idx].mean() - corrupt[idx].mean()
    if abs(denom) < 1e-9:
        return np.nan
    return float((patched[idx].mean() - corrupt[idx].mean()) / denom)

def recovery_diff(clean, corrupt, a, b, idx=None):
    return recovery(clean, corrupt, a, idx) - recovery(clean, corrupt, b, idx)

def bootstrap_recovery(clean, corrupt, patched, n_boot=MECHANISM_BOOTSTRAP_N):
    clean = np.asarray(clean, dtype=float)
    corrupt = np.asarray(corrupt, dtype=float)
    patched = np.asarray(patched, dtype=float)
    rng = np.random.default_rng(SEED)
    vals = []
    n = len(clean)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        denom = clean[idx].mean() - corrupt[idx].mean()
        if abs(denom) > 1e-9:
            vals.append((patched[idx].mean() - corrupt[idx].mean()) / denom)
    vals = np.asarray(vals, dtype=float)
    point = recovery(clean, corrupt, patched)
    return float(point), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def bootstrap_recovery_diff(clean, corrupt, patched_a, patched_b, n_boot=MECHANISM_BOOTSTRAP_N, seed=SEED):
    """Bootstrap CI for recovery(patched_a) - recovery(patched_b) over items."""
    clean = np.asarray(clean, dtype=float)
    corrupt = np.asarray(corrupt, dtype=float)
    patched_a = np.asarray(patched_a, dtype=float)
    patched_b = np.asarray(patched_b, dtype=float)

    rng = np.random.default_rng(seed)
    n = len(clean)
    vals = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ra = recovery(clean, corrupt, patched_a, idx)
        rb = recovery(clean, corrupt, patched_b, idx)
        if np.isfinite(ra) and np.isfinite(rb):
            vals.append(ra - rb)

    vals = np.asarray(vals, dtype=float)
    point = recovery_diff(clean, corrupt, patched_a, patched_b)
    return {
        "source_minus_pair_boot_mean": float(vals.mean()) if len(vals) else np.nan,
        "source_minus_pair_ci_low": float(np.percentile(vals, 2.5)) if len(vals) else np.nan,
        "source_minus_pair_ci_high": float(np.percentile(vals, 97.5)) if len(vals) else np.nan,
        "source_minus_pair_boot_n": int(len(vals)),
        "source_minus_pair_point": float(point),
    }

def load_hooked_transformer_safe(spec):
    tl_name = spec.get("tl_name", spec["hf_id"])
    hf_id = spec["hf_id"]

    base_kwargs = dict(
        device=DEVICE,
        dtype=MECHANISM_DTYPE,
        default_prepend_bos=False,
    )

    # Pass a manually patched tokenizer when possible.
    try:
        safe_tok = load_safe_tokenizer(hf_id)
        return HookedTransformer.from_pretrained(
            tl_name,
            tokenizer=safe_tok,
            **base_kwargs,
        )
    except Exception as first_error:
        last_error = first_error

    # Newer TransformerLens versions may accept tokenizer_kwargs.
    try:
        return HookedTransformer.from_pretrained(
            tl_name,
            tokenizer_kwargs={"add_bos_token": False, "trust_remote_code": True},
            **base_kwargs,
        )
    except Exception as second_error:
        last_error = second_error

    # Fallback.
    try:
        model = HookedTransformer.from_pretrained(tl_name, **base_kwargs)
        model.tokenizer = patch_tokenizer_no_bos_if_needed(model.tokenizer)
        return model
    except Exception as third_error:
        last_error = third_error

    raise last_error

In [ ]:
# ============================================================
# 10. Mechanistic analysis function
# ============================================================

def run_mechanism_for_model(spec):
    print("\n" + "=" * 100)
    print("Mechanism model:", spec["model_key"], spec.get("tl_name", spec["hf_id"]))
    print("=" * 100)

    model = load_hooked_transformer_safe(spec)
    model.eval()
    model.tokenizer = patch_tokenizer_no_bos_if_needed(model.tokenizer)

    # No explicit BOS for consistent Qwen/Gemma/OLMo handling.
    USE_BOS = False

    n_layers = int(model.cfg.n_layers)
    n_heads = int(model.cfg.n_heads)

    def toks_no_bos(text):
        ids = model.to_tokens(str(text), prepend_bos=False).squeeze(0).tolist()
        if isinstance(ids, int):
            ids = [ids]
        return [int(x) for x in ids]

    def cont_ids(text):
        return toks_no_bos(" " + str(text).strip())

    def is_single_token(text):
        return len(cont_ids(text)) == 1

    def to_tokens(prompts):
        return model.to_tokens(list(prompts), prepend_bos=USE_BOS).to(DEVICE)

    def maybe_mech_chat_wrap(prompt):
        if spec.get("tuning") == "instruct" and hasattr(model.tokenizer, "apply_chat_template"):
            try:
                return model.tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt}],
                    tokenize=False,
                    add_generation_prompt=True,
                )
            except Exception:
                return prompt
        return prompt

    def logit_parts(logits_2d, target_ids, distractor_ids):
        logits_2d = logits_2d.float()
        b = logits_2d.shape[0]
        idx = torch.arange(b, device=logits_2d.device)
        t = logits_2d[idx, target_ids]
        d = logits_2d[idx, distractor_ids]
        diff = t - d
        return (
            t.detach().cpu().numpy(),
            d.detach().cpu().numpy(),
            diff.detach().cpu().numpy(),
        )

    def score_prompts(prompts, target_ids, distractor_ids):
        rows = []
        for st in range(0, len(prompts), MECHANISM_BATCH_SIZE):
            ed = min(st + MECHANISM_BATCH_SIZE, len(prompts))
            toks = to_tokens(prompts[st:ed])
            t = torch.tensor(target_ids[st:ed], dtype=torch.long, device=DEVICE)
            d = torch.tensor(distractor_ids[st:ed], dtype=torch.long, device=DEVICE)
            with torch.inference_mode():
                logits = model(toks)[:, -1, :]
            _, _, diff = logit_parts(logits, t, d)
            rows.extend(list(diff))
            del toks, t, d, logits
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return np.asarray(rows, dtype=float)

    # Build single-token valid items.
    valid_rows = []
    for _, item in mechanism_stimuli.iterrows():
        if not (is_single_token(item["word"]) and is_single_token(item["distractor"]) and is_single_token(item["target"])):
            continue
        raw_conflict = mechanism_prompt(item["word"], item["target"], item["word"])
        valid_rows.append({
            **item.to_dict(),
            "word_id": cont_ids(item["word"])[0],
            "distractor_id": cont_ids(item["distractor"])[0],
            "target_id": cont_ids(item["target"])[0],
            "conflict_prompt": maybe_mech_chat_wrap(raw_conflict),
        })

    valid = pd.DataFrame(valid_rows).reset_index(drop=True)
    if len(valid) < 10:
        raise RuntimeError(f"Too few valid single-token items for {spec['model_key']}: {len(valid)}")

    # Multi-neutral behavior and representative clean prompt selection.
    conflict_scores = score_prompts(
        valid["conflict_prompt"].tolist(),
        valid["target_id"].astype(int).tolist(),
        valid["distractor_id"].astype(int).tolist(),
    )

    neutral_rows = []
    neutral_prompts, nt, nd, meta = [], [], [], []
    for i, row in valid.iterrows():
        for nw in choose_mechanism_neutrals(i, row):
            raw = mechanism_prompt(nw, row["target"], nw)
            prompt = maybe_mech_chat_wrap(raw)
            neutral_prompts.append(prompt)
            nt.append(int(row["target_id"]))
            nd.append(int(row["distractor_id"]))
            meta.append((i, nw, prompt))

    neutral_scores = score_prompts(neutral_prompts, nt, nd)
    for score, (i, nw, prompt) in zip(neutral_scores, meta):
        neutral_rows.append({
            "valid_i": int(i),
            "neutral_word": nw,
            "neutral_prompt": prompt,
            "neutral_score": float(score),
        })
    neutral = pd.DataFrame(neutral_rows)

    item_effect_rows = []
    for i, row in valid.iterrows():
        sub = neutral[neutral["valid_i"] == i]
        item_effect_rows.append({
            "valid_i": int(i),
            "item_id": int(row["item_id"]),
            "word": row["word"],
            "target": row["target"],
            "distractor": row["distractor"],
            "conflict_score": float(conflict_scores[i]),
            "neutral_mean": float(sub["neutral_score"].mean()),
            "effect": float(sub["neutral_score"].mean() - conflict_scores[i]),
        })
    item_effects = pd.DataFrame(item_effect_rows)

    # Choose representative neutral prompt nearest the item neutral mean.
    patch_rows = []
    for i, row in valid.iterrows():
        sub = neutral[neutral["valid_i"] == i].copy()
        mean = item_effects.loc[item_effects["valid_i"] == i, "neutral_mean"].iloc[0]
        sub["abs_dev"] = (sub["neutral_score"] - mean).abs()
        chosen = sub.sort_values("abs_dev").iloc[0]
        clean_prompt = chosen["neutral_prompt"]
        corrupt_prompt = row["conflict_prompt"]
        patch_rows.append({
            "valid_i": int(i),
            "item_id": int(row["item_id"]),
            "word": row["word"],
            "target": row["target"],
            "distractor": row["distractor"],
            "clean_prompt": clean_prompt,
            "corrupt_prompt": corrupt_prompt,
            "neutral_word": chosen["neutral_word"],
            "len_clean": int(model.to_tokens(clean_prompt, prepend_bos=USE_BOS).shape[-1]),
            "len_corrupt": int(model.to_tokens(corrupt_prompt, prepend_bos=USE_BOS).shape[-1]),
        })
    patch_prompts = pd.DataFrame(patch_rows)

    eligible = np.where(
        (patch_prompts["len_clean"].values == patch_prompts["len_corrupt"].values) &
        (item_effects["effect"].values > 0)
    )[0].tolist()
    if len(eligible) < 10:
        eligible = np.where(patch_prompts["len_clean"].values == patch_prompts["len_corrupt"].values)[0].tolist()

    patch_idx = eligible[:min(PATCH_N_ITEMS, len(eligible))]
    patch_items = valid.iloc[patch_idx].reset_index(drop=True)
    patch_prompts = patch_prompts.iloc[patch_idx].reset_index(drop=True)

    clean_tokens = to_tokens(patch_prompts["clean_prompt"])
    corrupt_tokens = to_tokens(patch_prompts["corrupt_prompt"])
    target_ids = torch.tensor(patch_items["target_id"].astype(int).tolist(), device=DEVICE)
    distractor_ids = torch.tensor(patch_items["distractor_id"].astype(int).tolist(), device=DEVICE)

    with torch.inference_mode():
        clean_logits, clean_cache = model.run_with_cache(clean_tokens)
        corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_tokens)

    clean_t, clean_d, clean_diff = logit_parts(clean_logits[:, -1, :], target_ids, distractor_ids)
    corrupt_t, corrupt_d, corrupt_diff = logit_parts(corrupt_logits[:, -1, :], target_ids, distractor_ids)

    # Position groups.
    pos_groups = []
    seq_len = int(clean_tokens.shape[1])

    for i, row in patch_items.iterrows():
        c_clean = clean_tokens[i].tolist()
        c_corrupt = corrupt_tokens[i].tolist()
        changed = [j for j, (a, b) in enumerate(zip(c_clean, c_corrupt)) if int(a) != int(b)]

        definition_subject = [changed[0]] if len(changed) > 0 else []
        query_word = [changed[1]] if len(changed) > 1 else []

        target_subseq = cont_ids(row["target"])
        starts = find_subsequence(c_corrupt, target_subseq)
        definition_target = [p for p in expand_positions(starts, len(target_subseq)) if p < seq_len - 1]

        groups = {
            "definition_subject": definition_subject,
            "definition_target": definition_target,
            "query_word": query_word,
            "definition_subject__definition_target": sorted(set(definition_subject + definition_target)),
            "definition_subject__query_word": sorted(set(definition_subject + query_word)),
            "definition_target__query_word": sorted(set(definition_target + query_word)),
            "source_all": sorted(set(definition_subject + definition_target + query_word)),
            "final": [seq_len - 1],
        }
        pos_groups.append(groups)

    def apply_vector_patch(act, src, group):
        act = act.clone()
        for b in range(act.shape[0]):
            ps = pos_groups[b][group]
            if len(ps):
                act[b, ps, ...] = src[b, ps, ...]
        return act

    def vector_patch_hook(group):
        def fn(act, hook):
            return apply_vector_patch(act, clean_cache[hook.name], group)
        return fn

    def random_vector_patch_hook(group, perm):
        def fn(act, hook):
            src = clean_cache[hook.name]
            act = act.clone()
            for b in range(act.shape[0]):
                ps = pos_groups[b][group]
                if len(ps):
                    act[b, ps, ...] = src[int(perm[b].item()), ps, ...]
            return act
        return fn

    def head_patch_hook(head, group):
        def fn(act, hook):
            src = clean_cache[hook.name]
            act = act.clone()
            for b in range(act.shape[0]):
                ps = pos_groups[b][group]
                if len(ps):
                    act[b, ps, head, :] = src[b, ps, head, :]
            return act
        return fn

    def head_zero_hook(head, group):
        def fn(act, hook):
            act = act.clone()
            for b in range(act.shape[0]):
                ps = pos_groups[b][group]
                if len(ps):
                    act[b, ps, head, :] = 0.0
            return act
        return fn

    def mlp_patch_hook(group):
        def fn(act, hook):
            return apply_vector_patch(act, clean_cache[hook.name], group)
        return fn

    def mlp_zero_hook(group):
        def fn(act, hook):
            act = act.clone()
            for b in range(act.shape[0]):
                ps = pos_groups[b][group]
                if len(ps):
                    act[b, ps, :] = 0.0
            return act
        return fn

    def run_scores(hooks):
        with torch.inference_mode():
            logits = model.run_with_hooks(corrupt_tokens, return_type="logits", fwd_hooks=hooks)[:, -1, :]
        t, d, diff = logit_parts(logits, target_ids, distractor_ids)
        return t, d, diff

    # Source residual patches.
    patch_scores = {}
    source_rows = []

    for group in SOURCE_COMBO_GROUPS:
        for comp in RESIDUAL_COMPONENTS:
            for layer in tqdm(range(n_layers), desc=f"{spec['model_key']} source {group}/{comp}"):
                name = f"blocks.{layer}.{comp}"
                if name not in model.hook_dict:
                    continue
                t, d, diff = run_scores([(name, vector_patch_hook(group))])
                label = patch_label(layer, comp, group)
                patch_scores[label] = diff
                rec, lo, hi = bootstrap_recovery(clean_diff, corrupt_diff, diff)
                source_rows.append({
                    "model_key": spec["model_key"],
                    "patch_label": label,
                    "group": group,
                    "component": comp,
                    "layer": int(layer),
                    "norm_depth": float(layer / max(1, n_layers)),
                    "recovery": rec,
                    "ci_low": lo,
                    "ci_high": hi,
                })

    source_df = pd.DataFrame(source_rows)
    source_candidates = source_df[(source_df["group"] == "source_all") & (source_df["layer"] >= MIN_MECHANISM_LAYER)]
    best_source = source_candidates.sort_values("recovery", ascending=False).iloc[0]
    best_layer = int(best_source["layer"])
    best_comp = best_source["component"]
    best_source_label = best_source["patch_label"]
    source_scores = patch_scores[best_source_label]

    pair_candidates = []
    for pair in SOURCE_PAIR_GROUPS:
        label = patch_label(best_layer, best_comp, pair)
        if label in patch_scores:
            pair_candidates.append((recovery(clean_diff, corrupt_diff, patch_scores[label]), label, pair))
    pair_candidates = sorted(pair_candidates, reverse=True, key=lambda x: x[0])
    best_pair_rec, best_pair_label, best_pair_group = pair_candidates[0]
    pair_scores = patch_scores[best_pair_label]

    # Component scans.
    def is_nondegenerate_component(scan_type, layer):
        if scan_type == "writer_source_positions" and int(layer) < MIN_NONDEGENERATE_COMPONENT_LAYER:
            return False
        return True

    component_rows = []

    if SCAN_WRITERS:
        for layer in tqdm(range(n_layers), desc=f"{spec['model_key']} writer components"):
            hname = f"blocks.{layer}.attn.hook_z"
            if hname in model.hook_dict:
                for head in range(n_heads):
                    t, d, diff = run_scores([(hname, head_patch_hook(head, "source_all"))])
                    component_rows.append({
                        "model_key": spec["model_key"],
                        "scan_type": "writer_source_positions",
                        "component_type": "attn_head",
                        "layer": int(layer),
                        "head": int(head),
                        "name": f"L{layer}H{head}",
                        "recovery": recovery(clean_diff, corrupt_diff, diff),
                        "target_delta": float((t - corrupt_t).mean()),
                        "distractor_delta": float((d - corrupt_d).mean()),
                        "is_nondegenerate_component": is_nondegenerate_component("writer_source_positions", layer),
                    })
            mname = f"blocks.{layer}.hook_mlp_out"
            if mname in model.hook_dict:
                t, d, diff = run_scores([(mname, mlp_patch_hook("source_all"))])
                component_rows.append({
                    "model_key": spec["model_key"],
                    "scan_type": "writer_source_positions",
                    "component_type": "mlp",
                    "layer": int(layer),
                    "head": -1,
                    "name": f"L{layer}MLP",
                    "recovery": recovery(clean_diff, corrupt_diff, diff),
                    "target_delta": float((t - corrupt_t).mean()),
                    "distractor_delta": float((d - corrupt_d).mean()),
                    "is_nondegenerate_component": is_nondegenerate_component("writer_source_positions", layer),
                })

    if SCAN_READERS:
        for layer in tqdm(range(best_layer, n_layers), desc=f"{spec['model_key']} reader components"):
            hname = f"blocks.{layer}.attn.hook_z"
            if hname in model.hook_dict:
                for head in range(n_heads):
                    t, d, diff = run_scores([(hname, head_patch_hook(head, "final"))])
                    component_rows.append({
                        "model_key": spec["model_key"],
                        "scan_type": "reader_final_position",
                        "component_type": "attn_head",
                        "layer": int(layer),
                        "head": int(head),
                        "name": f"L{layer}H{head}",
                        "recovery": recovery(clean_diff, corrupt_diff, diff),
                        "target_delta": float((t - corrupt_t).mean()),
                        "distractor_delta": float((d - corrupt_d).mean()),
                        "is_nondegenerate_component": is_nondegenerate_component("reader_final_position", layer),
                    })
            mname = f"blocks.{layer}.hook_mlp_out"
            if mname in model.hook_dict:
                t, d, diff = run_scores([(mname, mlp_patch_hook("final"))])
                component_rows.append({
                    "model_key": spec["model_key"],
                    "scan_type": "reader_final_position",
                    "component_type": "mlp",
                    "layer": int(layer),
                    "head": -1,
                    "name": f"L{layer}MLP",
                    "recovery": recovery(clean_diff, corrupt_diff, diff),
                    "target_delta": float((t - corrupt_t).mean()),
                    "distractor_delta": float((d - corrupt_d).mean()),
                    "is_nondegenerate_component": is_nondegenerate_component("reader_final_position", layer),
                })

    component_df = pd.DataFrame(component_rows)

    # Mediation: source_all patch plus zero top reader components.
    mediation_rows = []
    source_hook = (f"blocks.{best_layer}.{best_comp}", vector_patch_hook("source_all"))
    source_t, source_d, source_diff = run_scores([source_hook])
    source_rec = recovery(clean_diff, corrupt_diff, source_diff)

    if RUN_MEDIATION_ABLATIONS and len(component_df):
        top_readers = (
            component_df[component_df["scan_type"] == "reader_final_position"]
            .sort_values("recovery", ascending=False)
            .head(TOP_COMPONENTS)
        )
        for _, r in tqdm(top_readers.iterrows(), total=len(top_readers), desc=f"{spec['model_key']} mediation"):
            if r["component_type"] == "attn_head":
                hook = (f"blocks.{int(r['layer'])}.attn.hook_z", head_zero_hook(int(r["head"]), "final"))
            else:
                hook = (f"blocks.{int(r['layer'])}.hook_mlp_out", mlp_zero_hook("final"))

            t, d, diff = run_scores([source_hook, hook])
            rec = recovery(clean_diff, corrupt_diff, diff)
            mediation_rows.append({
                "model_key": spec["model_key"],
                "zeroed_component": r["name"],
                "component_type": r["component_type"],
                "layer": int(r["layer"]),
                "head": int(r["head"]),
                "source_recovery": float(source_rec),
                "source_plus_zero_recovery": float(rec),
                "drop": float(source_rec - rec),
            })

    mediation_df = pd.DataFrame(mediation_rows)

    # Split validation.
    rng = np.random.default_rng(SEED)
    n_items = len(patch_items)
    indices = np.arange(n_items)

    def select_best_source(idx):
        choices = []
        candidates = source_df[(source_df["group"] == "source_all") & (source_df["layer"] >= MIN_MECHANISM_LAYER)]
        for _, row in candidates.iterrows():
            label = row["patch_label"]
            rec = recovery(clean_diff, corrupt_diff, patch_scores[label], idx)
            choices.append((rec, label, row["component"], int(row["layer"])))
        return sorted(choices, reverse=True, key=lambda x: x[0])[0]

    def select_best_pair(idx, component, layer):
        choices = []
        for pair in SOURCE_PAIR_GROUPS:
            label = patch_label(layer, component, pair)
            if label in patch_scores:
                choices.append((recovery(clean_diff, corrupt_diff, patch_scores[label], idx), label, pair))
        return sorted(choices, reverse=True, key=lambda x: x[0])[0]

    split_rows = []
    for split_i in range(N_SPLITS):
        perm = rng.permutation(indices)
        test = perm[:len(perm)//2]
        discovery = perm[len(perm)//2:]

        _, selected_source, selected_component, selected_layer = select_best_source(discovery)
        _, selected_pair, pair_group = select_best_pair(discovery, selected_component, selected_layer)

        split_rows.append({
            "model_key": spec["model_key"],
            "split": int(split_i),
            "selected_source": selected_source,
            "selected_pair": selected_pair,
            "pair_group": pair_group,
            "test_source_minus_pair": recovery_diff(
                clean_diff,
                corrupt_diff,
                patch_scores[selected_source],
                patch_scores[selected_pair],
                test,
            ),
        })
    split_df = pd.DataFrame(split_rows)

    # Random-pair controls.
    random_rows = []
    source_name = f"blocks.{best_layer}.{best_comp}"
    for pi in range(N_RANDOM_PERMUTATIONS):
        gen = torch.Generator(device=DEVICE)
        gen.manual_seed(SEED + 1000 + pi)
        perm = torch.randperm(n_items, device=DEVICE, generator=gen)
        t, d, diff = run_scores([(source_name, random_vector_patch_hook("source_all", perm))])
        random_rows.append({
            "model_key": spec["model_key"],
            "perm": int(pi),
            "matched_recovery": recovery(clean_diff, corrupt_diff, source_scores),
            "random_recovery": recovery(clean_diff, corrupt_diff, diff),
        })
    random_df = pd.DataFrame(random_rows)

    # Definition-target swap control.
    # This is computed inside the final mechanism function so it reuses the exact
    # clean_cache, corrupt_tokens, position groups, hook, and normalization used
    # for the main source-triplet result. Therefore the matched baseline is the
    # same quantity as source_rec / best_source_recovery by construction.
    def_target_swap_rows = []
    def_target_swap_item_summary_rows = []
    def_target_swap_summary = pd.DataFrame()

    if RUN_DEFINITION_TARGET_SWAP_CONTROL:
        swap_diffs = []
        swap_targets = []
        swap_distractors = []
        rng_swap = np.random.default_rng(SEED + 7000 + sum(ord(ch) for ch in spec["model_key"]))

        for swap_i in tqdm(range(N_DEFINITION_TARGET_SWAPS), desc=f"{spec['model_key']} def-target swaps"):
            donors = np.empty(n_items, dtype=int)
            for b in range(n_items):
                choices = [j for j in range(n_items) if j != b]
                donors[b] = int(rng_swap.choice(choices))

            def def_target_swap_hook(act, hook, donors=donors):
                src = clean_cache[hook.name]
                act = act.clone()
                for b in range(act.shape[0]):
                    # Keep identity positions matched to the current item.
                    for group in ["definition_subject", "query_word"]:
                        ps = pos_groups[b][group]
                        if len(ps):
                            act[b, ps, ...] = src[b, ps, ...]

                    # Swap only the local value position from a donor item.
                    dst_ps = pos_groups[b]["definition_target"]
                    donor = int(donors[b])
                    src_ps = pos_groups[donor]["definition_target"]
                    if len(dst_ps) and len(src_ps):
                        m = min(len(dst_ps), len(src_ps))
                        act[b, dst_ps[:m], ...] = src[donor, src_ps[:m], ...]
                return act

            swap_t, swap_d, swap_diff = run_scores([(source_name, def_target_swap_hook)])
            swap_diffs.append(swap_diff)
            swap_targets.append(swap_t)
            swap_distractors.append(swap_d)

            for b in range(n_items):
                def_target_swap_rows.append({
                    "model_key": spec["model_key"],
                    "model": spec.get("label", spec["model_key"]),
                    "item_id": int(patch_items.iloc[b]["item_id"]),
                    "word": patch_items.iloc[b]["word"],
                    "target": patch_items.iloc[b]["target"],
                    "distractor": patch_items.iloc[b]["distractor"],
                    "neutral_word": patch_prompts.iloc[b]["neutral_word"],
                    "hook_name": source_name,
                    "triplet_selected_layer": int(best_layer),
                    "control_type": "definition_target_swap",
                    "swap_i": int(swap_i),
                    "donor_row_index": int(donors[b]),
                    "donor_item_id": int(patch_items.iloc[donors[b]]["item_id"]),
                    "donor_word": patch_items.iloc[donors[b]]["word"],
                    "donor_target": patch_items.iloc[donors[b]]["target"],
                    "m_clean": float(clean_diff[b]),
                    "m_corrupt": float(corrupt_diff[b]),
                    "m_matched": float(source_scores[b]),
                    "m_patched": float(swap_diff[b]),
                    "target_logit_delta": float(swap_t[b] - corrupt_t[b]),
                    "distractor_logit_delta": float(swap_d[b] - corrupt_d[b]),
                })

        swap_diffs = np.asarray(swap_diffs, dtype=float)
        swap_mean_diff = swap_diffs.mean(axis=0)
        swap_global_recovery = recovery(clean_diff, corrupt_diff, swap_mean_diff)
        matched_global_recovery = source_rec
        matched_minus_swap = matched_global_recovery - swap_global_recovery

        for b in range(n_items):
            item_den = float(clean_diff[b] - corrupt_diff[b])
            matched_item_rec = np.nan if abs(item_den) < 1e-9 else float((source_scores[b] - corrupt_diff[b]) / item_den)
            swap_item_rec = np.nan if abs(item_den) < 1e-9 else float((swap_mean_diff[b] - corrupt_diff[b]) / item_den)
            def_target_swap_item_summary_rows.append({
                "model_key": spec["model_key"],
                "model": spec.get("label", spec["model_key"]),
                "item_id": int(patch_items.iloc[b]["item_id"]),
                "word": patch_items.iloc[b]["word"],
                "target": patch_items.iloc[b]["target"],
                "distractor": patch_items.iloc[b]["distractor"],
                "neutral_word": patch_prompts.iloc[b]["neutral_word"],
                "hook_name": source_name,
                "triplet_selected_layer": int(best_layer),
                "m_clean": float(clean_diff[b]),
                "m_corrupt": float(corrupt_diff[b]),
                "m_matched": float(source_scores[b]),
                "m_swap_mean": float(swap_mean_diff[b]),
                "matched_item_recovery": matched_item_rec,
                "swap_item_recovery_mean": swap_item_rec,
                "matched_minus_swap_margin": float(source_scores[b] - swap_mean_diff[b]),
                "swap_below_matched_margin": bool(swap_mean_diff[b] < source_scores[b]),
                "n_swaps_per_item": int(N_DEFINITION_TARGET_SWAPS),
            })

        def_target_swap_summary = pd.DataFrame([{
            "model_key": spec["model_key"],
            "model": spec.get("label", spec["model_key"]),
            "n_items": int(n_items),
            "hook_name": source_name,
            "triplet_selected_layer": int(best_layer),
            "matched_recovery_global": float(matched_global_recovery),
            "swap_recovery_global": float(swap_global_recovery),
            "matched_minus_swap_global": float(matched_minus_swap),
            "fraction_items_swap_below_matched": float(np.mean(swap_mean_diff < source_scores)),
            "n_swaps_per_item": int(N_DEFINITION_TARGET_SWAPS),
        }])

    definition_target_swap_df = pd.DataFrame(def_target_swap_rows)
    definition_target_swap_item_summary_df = pd.DataFrame(def_target_swap_item_summary_rows)

    # Logit decomposition.
    source_t, source_d, source_diff = run_scores([(source_name, vector_patch_hook("source_all"))])
    logit_decomp = pd.DataFrame([{
        "model_key": spec["model_key"],
        "best_source_label": best_source_label,
        "target_logit_delta": float((source_t - corrupt_t).mean()),
        "distractor_logit_delta": float((source_d - corrupt_d).mean()),
        "logit_diff_delta": float((source_diff - corrupt_diff).mean()),
    }])

    summary = {
        "model_key": spec["model_key"],
        "family": spec["family"],
        "params_b": spec["params_b"],
        "tuning": spec["tuning"],
        "n_layers": int(n_layers),
        "n_heads": int(n_heads),
        "n_patching_items": int(n_items),
        "best_source_label": best_source_label,
        "best_source_layer": int(best_layer),
        "best_source_component": best_comp,
        "best_source_recovery": float(recovery(clean_diff, corrupt_diff, source_scores)),
        "best_pair_label": best_pair_label,
        "best_pair_group": best_pair_group,
        "best_pair_recovery": float(recovery(clean_diff, corrupt_diff, pair_scores)),
        "source_minus_pair": float(recovery_diff(clean_diff, corrupt_diff, source_scores, pair_scores)),
        "split_mean_source_minus_pair": float(split_df["test_source_minus_pair"].mean()),
        "split_positive_fraction": float((split_df["test_source_minus_pair"] > 0).mean()),
        "random_mean": float(random_df["random_recovery"].mean()),
        "random_max": float(random_df["random_recovery"].max()),
    }

    patch_items_out = patch_items.assign(model_key=spec["model_key"])
    patch_prompts_out = patch_prompts.assign(model_key=spec["model_key"])
    item_effects_out = item_effects.assign(model_key=spec["model_key"])

    del clean_cache, corrupt_cache, clean_logits, corrupt_logits, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "summary": summary,
        "source_patches": source_df,
        "component_scan": component_df,
        "mediation": mediation_df,
        "split": split_df,
        "random": random_df,
        "logit_decomp": logit_decomp,
        "definition_target_swap": definition_target_swap_df,
        "definition_target_swap_item_summary": definition_target_swap_item_summary_df,
        "definition_target_swap_summary": def_target_swap_summary,
        "patch_items": patch_items_out,
        "patch_prompts": patch_prompts_out,
        "item_effects": item_effects_out,
    }

In [ ]:
# ============================================================
# 11. Run mechanistic analyses
# ============================================================
mechanism_outputs = {}
mechanism_run_log = []

if RUN_MECHANISM:
    for spec in MECHANISM_MODEL_SPECS:
        if not spec.get("enabled", True):
            continue
        try:
            out = run_mechanism_for_model(spec)
            mechanism_outputs[spec["model_key"]] = out
            mechanism_run_log.append({**spec, "status": "success", "error": ""})
        except Exception as e:
            print("Mechanism model failed:", spec["model_key"], repr(e))
            mechanism_run_log.append({**spec, "status": "error", "error": repr(e)})
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
else:
    print("RUN_MECHANISM is False; skipping mechanism.")

mechanism_run_log_df = pd.DataFrame(mechanism_run_log)
mechanism_summary_df = pd.DataFrame([v["summary"] for v in mechanism_outputs.values()]) if mechanism_outputs else pd.DataFrame()
mechanism_source_df = pd.concat([v["source_patches"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs else pd.DataFrame()
mechanism_component_df = pd.concat([v["component_scan"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs else pd.DataFrame()
mechanism_component_nondeg_df = mechanism_component_df[mechanism_component_df.get("is_nondegenerate_component", True) == True].copy() if len(mechanism_component_df) else pd.DataFrame()
mechanism_mediation_df = pd.concat([v["mediation"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs else pd.DataFrame()
mechanism_split_df = pd.concat([v["split"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs else pd.DataFrame()
mechanism_random_df = pd.concat([v["random"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs else pd.DataFrame()
mechanism_logit_df = pd.concat([v["logit_decomp"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs else pd.DataFrame()
mechanism_def_target_swap_df = pd.concat([v["definition_target_swap"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs and any("definition_target_swap" in v for v in mechanism_outputs.values()) else pd.DataFrame()
mechanism_def_target_swap_item_summary_df = pd.concat([v["definition_target_swap_item_summary"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs and any("definition_target_swap_item_summary" in v for v in mechanism_outputs.values()) else pd.DataFrame()
mechanism_def_target_swap_summary_df = pd.concat([v["definition_target_swap_summary"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs and any("definition_target_swap_summary" in v for v in mechanism_outputs.values()) else pd.DataFrame()
mechanism_patch_items_df = pd.concat([v["patch_items"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs else pd.DataFrame()
mechanism_patch_prompts_df = pd.concat([v["patch_prompts"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs else pd.DataFrame()
mechanism_item_effects_df = pd.concat([v["item_effects"] for v in mechanism_outputs.values()], ignore_index=True) if mechanism_outputs else pd.DataFrame()

safe_to_csv(mechanism_run_log_df, "mechanism_model_run_log.csv")
if len(mechanism_summary_df): safe_to_csv(mechanism_summary_df, "mechanism_summary.csv")
if len(mechanism_source_df): safe_to_csv(mechanism_source_df, "mechanism_source_residual_patches.csv")
if len(mechanism_component_df): safe_to_csv(mechanism_component_df, "mechanism_component_scan.csv")
if len(mechanism_component_nondeg_df): safe_to_csv(mechanism_component_nondeg_df, "mechanism_component_scan_nondegenerate.csv")
if len(mechanism_mediation_df): safe_to_csv(mechanism_mediation_df, "mechanism_component_mediation_zero_ablation.csv")
if len(mechanism_split_df): safe_to_csv(mechanism_split_df, "mechanism_split_validation.csv")
if len(mechanism_random_df): safe_to_csv(mechanism_random_df, "mechanism_random_controls.csv")
if len(mechanism_logit_df): safe_to_csv(mechanism_logit_df, "mechanism_logit_decomposition.csv")
if len(mechanism_def_target_swap_df): safe_to_csv(mechanism_def_target_swap_df, "mechanism_definition_target_swap_controls.csv")
if len(mechanism_def_target_swap_item_summary_df): safe_to_csv(mechanism_def_target_swap_item_summary_df, "mechanism_definition_target_swap_item_summary.csv")
if len(mechanism_def_target_swap_summary_df): safe_to_csv(mechanism_def_target_swap_summary_df, "mechanism_definition_target_swap_summary.csv")
if len(mechanism_patch_items_df): safe_to_csv(mechanism_patch_items_df, "mechanism_patch_items.csv")
if len(mechanism_patch_prompts_df): safe_to_csv(mechanism_patch_prompts_df, "mechanism_patch_prompts.csv")
if len(mechanism_item_effects_df): safe_to_csv(mechanism_item_effects_df, "mechanism_item_effects.csv")

display(mechanism_run_log_df)
display(mechanism_summary_df)
display(mechanism_component_nondeg_df.sort_values("recovery", ascending=False).head(20) if len(mechanism_component_nondeg_df) else mechanism_component_nondeg_df)
display(mechanism_mediation_df.sort_values("drop", ascending=False).head(20) if len(mechanism_mediation_df) else mechanism_mediation_df)

In [ ]:
# ============================================================
# 12. Mechanistic summaries and figures
# ============================================================
if RUN_MECHANISM and len(mechanism_summary_df):
    # Derived split summary.
    split_summary = mechanism_split_df.groupby("model_key", as_index=False).agg(
        split_mean_source_minus_pair=("test_source_minus_pair", "mean"),
        split_ci_low=("test_source_minus_pair", lambda x: float(np.percentile(x, 2.5))),
        split_ci_high=("test_source_minus_pair", lambda x: float(np.percentile(x, 97.5))),
        split_positive_fraction=("test_source_minus_pair", lambda x: float((np.asarray(x) > 0).mean())),
    )
    safe_to_csv(split_summary, "mechanism_split_validation_summary.csv")

    random_summary = mechanism_random_df.groupby("model_key", as_index=False).agg(
        matched_recovery=("matched_recovery", "mean"),
        random_mean=("random_recovery", "mean"),
        random_min=("random_recovery", "min"),
        random_max=("random_recovery", "max"),
    )
    safe_to_csv(random_summary, "mechanism_random_controls_summary.csv")

    # Evidence checklist.
    checklist_rows = []
    checklist_rows.append({
        "area": "source_triplet_advantage",
        "passed": bool((mechanism_summary_df["source_minus_pair"] > 0).all()),
        "evidence": f"{int((mechanism_summary_df['source_minus_pair'] > 0).sum())}/{len(mechanism_summary_df)} models positive",
    })
    checklist_rows.append({
        "area": "split_validation_base_stability",
        "passed": bool((mechanism_summary_df["split_mean_source_minus_pair"] > 0).all()),
        "evidence": mechanism_summary_df[["model_key", "split_mean_source_minus_pair", "split_positive_fraction"]].to_dict("records"),
    })
    checklist_rows.append({
        "area": "random_controls",
        "passed": bool((mechanism_summary_df["random_max"] < mechanism_summary_df["best_source_recovery"]).all()),
        "evidence": mechanism_summary_df[["model_key", "best_source_recovery", "random_mean", "random_max"]].to_dict("records"),
    })
    checklist_rows.append({
        "area": "component_scan_available",
        "passed": bool(len(mechanism_component_nondeg_df) > 0),
        "evidence": "Non-degenerate component scan table generated.",
    })
    checklist_rows.append({
        "area": "component_mediation_available",
        "passed": bool(len(mechanism_mediation_df) and (mechanism_mediation_df["drop"] > 0).any()),
        "evidence": "Reader zero-ablation mediation drops generated.",
    })
    checklist_rows.append({
        "area": "distractor_suppression",
        "passed": bool(len(mechanism_logit_df) and (mechanism_logit_df["distractor_logit_delta"] < mechanism_logit_df["target_logit_delta"]).all()),
        "evidence": mechanism_logit_df.to_dict("records") if len(mechanism_logit_df) else "missing",
    })
    mechanism_checklist_df = pd.DataFrame(checklist_rows)
    safe_to_csv(mechanism_checklist_df, "mechanism_evidence_checklist.csv")

    display(split_summary)
    display(random_summary)
    display(mechanism_checklist_df)

    # Figures.
    y = np.arange(len(mechanism_summary_df))
    plot = mechanism_summary_df.sort_values("model_key")
    plt.figure(figsize=(8, max(4, 0.4 * len(plot) + 2)))
    plt.scatter(plot["source_minus_pair"], y)
    plt.axvline(0, linewidth=1)
    plt.yticks(y, plot["model_key"])
    plt.xlabel("source_all recovery - best_pair recovery")
    plt.title("Source-triplet advantage")
    savefig("mechanism_source_all_minus_best_pair.png")

    x = np.arange(len(plot))
    plt.figure(figsize=(9, 4))
    plt.bar(x - 0.2, plot["best_source_recovery"], width=0.4, label="source_all")
    plt.bar(x + 0.2, plot["best_pair_recovery"], width=0.4, label="best pair")
    plt.axhline(0, linewidth=1)
    plt.xticks(x, plot["model_key"], rotation=30, ha="right")
    plt.ylabel("Recovery")
    plt.title("source_all vs strongest pair")
    plt.legend()
    savefig("mechanism_source_all_vs_best_pair.png")

    if len(mechanism_component_nondeg_df):
        for model_key, sub in mechanism_component_nondeg_df.groupby("model_key"):
            top = sub.sort_values("recovery", ascending=False).head(25)
            plt.figure(figsize=(10, 5))
            x = np.arange(len(top))
            plt.bar(x, top["recovery"])
            plt.axhline(0, linewidth=1)
            labels = top["scan_type"].str.replace("_", " ") + "\n" + top["name"]
            plt.xticks(x, labels, rotation=90, fontsize=7)
            plt.ylabel("Recovery")
            plt.title(f"Top non-degenerate component patches: {model_key}")
            savefig(f"mechanism_top_components_{model_key}.png")

    if len(mechanism_mediation_df):
        for model_key, sub in mechanism_mediation_df.groupby("model_key"):
            top = sub.sort_values("drop", ascending=False).head(20)
            plt.figure(figsize=(9, 4))
            x = np.arange(len(top))
            plt.bar(x, top["drop"])
            plt.axhline(0, linewidth=1)
            plt.xticks(x, top["zeroed_component"], rotation=90, fontsize=8)
            plt.ylabel("Drop from source_all recovery")
            plt.title(f"Reader mediation drops: {model_key}")
            savefig(f"mechanism_mediation_drop_{model_key}.png")

    if len(mechanism_logit_df):
        x = np.arange(len(mechanism_logit_df))
        plt.figure(figsize=(8, 4))
        plt.bar(x - 0.2, mechanism_logit_df["target_logit_delta"], width=0.4, label="target logit delta")
        plt.bar(x + 0.2, mechanism_logit_df["distractor_logit_delta"], width=0.4, label="distractor logit delta")
        plt.axhline(0, linewidth=1)
        plt.xticks(x, mechanism_logit_df["model_key"], rotation=30, ha="right")
        plt.ylabel("Mean logit delta under best source_all patch")
        plt.title("Target-vs-distractor logit decomposition")
        plt.legend()
        savefig("mechanism_target_vs_distractor_decomposition.png")
else:
    print("Mechanistic summaries skipped.")

In [ ]:
# ============================================================
# 13. Manuscript-ready tables
# ============================================================

tables = {}

if len(behavior_effects_df):
    tables["Table_Behavior_Model_Level_Sum"] = behavior_summary_model[behavior_summary_model["score_mode"] == "sum_logprob"].copy()
    tables["Table_Behavior_Aggregate_Conflict_Type_Sum"] = behavior_aggregate_type[behavior_aggregate_type["score_mode"] == "sum_logprob"].copy()
    tables["Table_Behavior_Aggregate_Prompt_Style_Sum"] = behavior_aggregate_style[behavior_aggregate_style["score_mode"] == "sum_logprob"].copy()

    main_reg = behavior_regression_df[
        (behavior_regression_df["regression"] == "main_controls") &
        (behavior_regression_df["term"].isin(["Intercept", "lexical_advantage"]))
    ].copy()
    tables["Table_Behavior_Main_Control_Regression"] = main_reg

if len(mechanism_summary_df):
    tables["Table_Mechanism_Source_Triplet"] = mechanism_summary_df.copy()
    tables["Table_Mechanism_Random_Controls"] = random_summary.copy() if "random_summary" in globals() else pd.DataFrame()
    tables["Table_Mechanism_Top_Mediation_Drops"] = (
        mechanism_mediation_df.sort_values("drop", ascending=False).head(12).copy()
        if len(mechanism_mediation_df) else pd.DataFrame()
    )
    tables["Table_Mechanism_Logit_Decomposition"] = mechanism_logit_df.copy()

for name, df in tables.items():
    safe_to_csv(df, f"manuscript_{name}.csv")

# Create a compact Markdown summary.
summary_parts = []
summary_parts.append("# Semantic Stroop Final Reproduction Summary\n")
summary_parts.append(f"Run ID: `{RUN_ID}`\n")
summary_parts.append(f"Output directory: `{OUT_DIR}`\n")

if "behavior_run_log_df" in globals() and len(behavior_run_log_df):
    summary_parts.append("\n## Behavior run log\n")
    summary_parts.append(behavior_run_log_df.to_markdown(index=False))

if "behavior_summary_model" in globals() and len(behavior_summary_model):
    summary_parts.append("\n## Behavior model-level summary, summed scoring\n")
    summary_parts.append(behavior_summary_model[behavior_summary_model["score_mode"] == "sum_logprob"].to_markdown(index=False))

if "behavior_aggregate_type" in globals() and len(behavior_aggregate_type):
    summary_parts.append("\n## Aggregate conflict-family summary, summed scoring\n")
    summary_parts.append(behavior_aggregate_type[behavior_aggregate_type["score_mode"] == "sum_logprob"].to_markdown(index=False))

if "behavior_aggregate_style" in globals() and len(behavior_aggregate_style):
    summary_parts.append("\n## Aggregate prompt-style summary, summed scoring\n")
    summary_parts.append(behavior_aggregate_style[behavior_aggregate_style["score_mode"] == "sum_logprob"].to_markdown(index=False))

if "behavior_regression_df" in globals() and len(behavior_regression_df):
    summary_parts.append("\n## Main-control regression terms\n")
    summary_parts.append(
        behavior_regression_df[
            (behavior_regression_df["regression"] == "main_controls") &
            (behavior_regression_df["term"].isin(["Intercept", "lexical_advantage", "is_instruct", "log_params_b"]))
        ].to_markdown(index=False)
    )


if "behavior_regression_robustness_df" in globals() and len(behavior_regression_robustness_df):
    summary_parts.append("\n## Regression robustness specifications\n")
    summary_parts.append(behavior_regression_robustness_df.to_markdown(index=False))

if "mechanism_run_log_df" in globals() and len(mechanism_run_log_df):
    summary_parts.append("\n## Mechanism run log\n")
    summary_parts.append(mechanism_run_log_df.to_markdown(index=False))

if "mechanism_summary_df" in globals() and len(mechanism_summary_df):
    summary_parts.append("\n## Mechanism source-triplet summary\n")
    summary_parts.append(mechanism_summary_df.to_markdown(index=False))

if "mechanism_logit_df" in globals() and len(mechanism_logit_df):
    summary_parts.append("\n## Mechanism logit decomposition\n")
    summary_parts.append(mechanism_logit_df.to_markdown(index=False))

if "behavior_checklist_df" in globals() and len(behavior_checklist_df):
    summary_parts.append("\n## Behavior evidence checklist\n")
    summary_parts.append(behavior_checklist_df.to_markdown(index=False))

if "mechanism_checklist_df" in globals() and len(mechanism_checklist_df):
    summary_parts.append("\n## Mechanism evidence checklist\n")
    summary_parts.append(mechanism_checklist_df.to_markdown(index=False))

summary_md = "\n\n".join(summary_parts)
summary_path = SUMMARY_DIR / "final_reproduction_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
display(Markdown(summary_md[:20000] + ("\n\n...summary truncated..." if len(summary_md) > 20000 else "")))
print("Saved:", summary_path)

In [ ]:
# ============================================================
# 14. Archive outputs
# ============================================================
readme = f"""
Semantic Stroop Final Reproduction Outputs

Run ID: {RUN_ID}

Main directories:
- csv/: all generated tables and intermediate CSVs
- figures/: all generated PNG figures
- summaries/: Markdown summary files

Main behavior outputs:
- behavior_model_run_log.csv
- behavior_external_validity_stimuli.csv
- behavior_item_prompt_effects.csv
- behavior_summary_by_model.csv
- behavior_summary_by_model_conflict_type.csv
- behavior_summary_by_model_prompt_style.csv
- behavior_aggregate_by_conflict_type.csv
- behavior_aggregate_by_prompt_style.csv
- behavior_control_regressions.csv
- behavior_regression_robustness.csv
- behavior_evidence_checklist.csv

Main mechanism outputs:
- mechanism_model_run_log.csv
- mechanism_summary.csv
- mechanism_source_residual_patches.csv
- mechanism_component_scan.csv
- mechanism_component_scan_nondegenerate.csv
- mechanism_component_mediation_zero_ablation.csv
- mechanism_split_validation.csv
- mechanism_random_controls.csv
- mechanism_logit_decomposition.csv
- mechanism_definition_target_swap_controls.csv
- mechanism_definition_target_swap_item_summary.csv
- mechanism_definition_target_swap_summary.csv
- mechanism_evidence_checklist.csv

Manuscript-ready outputs:
- manuscript_Table_*.csv
- summaries/final_reproduction_summary.md

Figures:
- behavior_model_forest_*.png
- behavior_conflict_type_heatmap_*.png
- behavior_prompt_style_heatmap_*.png
- behavior_lexical_prior_vs_effect_*.png
- behavior_base_vs_instruct_*.png
- mechanism_source_all_minus_best_pair.png
- mechanism_source_all_vs_best_pair.png
- mechanism_top_components_*.png
- mechanism_mediation_drop_*.png
- mechanism_target_vs_distractor_decomposition.png
"""

(OUT_DIR / "README.txt").write_text(readme, encoding="utf-8")

zip_path = shutil.make_archive(str(OUT_DIR), "zip", root_dir=OUT_DIR)
print("Created archive:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Archive saved at:", zip_path)